# 3D reporter timelapse — 05c_reporter_temporal_ordering

**Feeds:** Fig 5h

**Position in the chain:** run the numbered notebooks in order

Ported unchanged from the original analysis: outputs are as they ran, and no code
cell was edited. Paths appear as `<analysis-root>/...`.


# Reporter Temporal Ordering

This notebook extracts Theme 2 from the broader cooperativity exploration.

The goal here is narrower:

- establish what the direct abundance and thresholded population traces look like in clock time
- quantify whether `RFP` precedes `YFP` position by position
- test how sensitive that timing story is to threshold choice

Notebook `05b` remains the Theme 1 spatial-context notebook.


## Interpretation Frame

This notebook is about sequence, not mechanism.

The questions are:

1. what do the abundance, domain-size, and threshold-free intensity traces look like in clock time?
2. does `RFP` typically precede `YFP` position by position?
3. does that ordering remain stable when the thresholded reporter definitions change?

To keep reruns lighter, the per-position half-max tables are stored as shared cooperativity caches in `results/tables`.


## Upstream Signal Preprocessing

The thresholded reporter metrics shown here are inherited from notebook `05`; this notebook does **not** reprocess images from scratch.

The upstream preprocessing chain is:

1. apply reporter-specific illumination correction to the raw `FOXF1-RFP` and `BMP4-YFP` images using the shared channel-wide illumination fields estimated earlier in the pipeline
2. for each image/frame separately, measure the whole off-cyst pixel median **after** illumination correction and subtract that per-image scalar background
3. after that subtraction, estimate the reporter-negative within-cyst baseline and sigma from early corrected cyst pixels, then call positive pixels at the chosen `N sigma` threshold

So the thresholded areas, positive fractions, and positive-region intensity summaries in these notebooks all sit on top of:

- illumination correction
- per-image off-cyst median background subtraction
- within-cyst baseline / sigma estimation for thresholding

By default in the manuscript-facing reporter notebooks, the thresholded positive-fraction comparisons use:

- `FOXF1-RFP`: `4 sigma`
- `BMP4-YFP`: `3 sigma`


## Setup


In [ ]:
import sys
from pathlib import Path

import itertools
import shutil
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from scipy import stats

pd.set_option("display.max_columns", 200)
plt.rcParams["figure.dpi"] = 120


In [ ]:
cwd = Path.cwd().resolve()
root_candidates = [cwd] + list(cwd.parents[:3])
ROOT = None
for candidate in root_candidates:
    if (candidate / "data").exists() and (candidate / "results").exists():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Could not locate project root from current working directory.")

qc_dir = ROOT / "scripts" / "qc"
if str(qc_dir) not in sys.path:
    sys.path.insert(0, str(qc_dir))

from notebook_display_helpers import (
    TIME_DISPLAY_OFFSET_HOURS,
    display_time_hours as _display_time_hours,
    format_display_hours as _format_display_hours,
    format_display_hour_range as _format_display_hour_range,
    offset_time_hours_df as _offset_time_hours_df,
    set_display_time_axis as _set_display_time_axis,
    set_display_time_colorbar as _set_display_time_colorbar,
)
from notebook_figure_helpers import (
    NotebookFigureExportPaths,
    install_png_vector_savefig_exports,
)
from notebook_invariant_helpers import (
    THEME_DEFAULT_SIGMA_BY_REPORTER,
    assert_metric_name_sigma_consistency,
    assert_no_excluded_keys_in_analysis,
    assert_theme_default_sigmas,
)
from reporter_cooperativity_shared import (
    COMMON_HALFMAX_METRICS,
    METRIC_LABELS,
    REPORTER_COLORS,
    aggregate_mean_trace,
    build_halfmax_lookup,
    first_crossing_time,
    focus_ylim_from_arrays,
    gaussian_smooth,
    halfmax_time_for_position,
    halfmax_time_from_lookup,
    local_gradient,
    load_basic_inputs,
    load_or_build_halfmax_cache,
    normalize_trace,
)

ROOT, population_metrics, global_thresholds = load_basic_inputs(ROOT)
install_png_vector_savefig_exports()
FIGURE_EXPORTS = NotebookFigureExportPaths.create(
    ROOT,
    "05c",
    alternate_strip_prefixes=("05c_",),
    candidate_strip_prefixes=("05c_", "main_candidate_"),
)
FIGURE_DIR = FIGURE_EXPORTS.figure_dir
ALTERNATE_FIGURE_DIR = FIGURE_EXPORTS.alternate_dir
CANDIDATE_FIGURE_DIR = FIGURE_EXPORTS.candidate_dir
TABLE_DIR = ROOT / "results" / "tables"
FRAME_METRICS_PATH = TABLE_DIR / "05_reporter_metrics_by_frame.tsv"
HALFMAX_CACHE_PATH = TABLE_DIR / "05_cooperativity_halfmax_times.tsv"
DEFAULT_RFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["RFP"])
DEFAULT_YFP_SIGMA = int(THEME_DEFAULT_SIGMA_BY_REPORTER["YFP"])
DEFAULT_POSITIVE_FRACTION_METRICS = {
    "RFP": f"positive_fraction_sigma{DEFAULT_RFP_SIGMA}",
    "YFP": f"positive_fraction_sigma{DEFAULT_YFP_SIGMA}",
}
DEFAULT_POSITIVE_MEAN_METRICS = {
    "RFP": f"positive_mean_intensity_sigma{DEFAULT_RFP_SIGMA}_z",
    "YFP": f"positive_mean_intensity_sigma{DEFAULT_YFP_SIGMA}_z",
}
THRESHOLD_SENSITIVITY_SIGMAS = list(range(1, 9))
REPORTER_COLORS = {
    "RFP": "#d62728",
    "YFP": "#d8a106",
}
REPORTER_FILL_ALPHA = 0.18
DISPLAY_REPORTER_LABELS = {
    "RFP": "FOXF1-RFP",
    "YFP": "BMP4-YFP",
}
TABLE_DIR.mkdir(parents=True, exist_ok=True)
frame_metrics = pd.read_csv(
    FRAME_METRICS_PATH,
    sep="\t",
    low_memory=False,
    usecols=["position_label", "time_index", "reporter", "organoid_area_px", "exclude_from_analysis"],
)

assert_theme_default_sigmas(
    {"RFP": DEFAULT_RFP_SIGMA, "YFP": DEFAULT_YFP_SIGMA},
    context="05c Theme 2 default thresholds",
)
assert_metric_name_sigma_consistency(
    DEFAULT_POSITIVE_FRACTION_METRICS,
    context="05c default positive-fraction pairing",
)
assert_metric_name_sigma_consistency(
    DEFAULT_POSITIVE_MEAN_METRICS,
    context="05c default positive-mean pairing",
)
assert_no_excluded_keys_in_analysis(
    frame_metrics,
    population_metrics,
    key_columns=("position_label", "time_index", "reporter"),
    context="05c population metrics",
)


def alternate_figure_path(filename: str | Path) -> Path:
    return FIGURE_EXPORTS.alternate_path(filename)


def candidate_figure_path(filename: str | Path) -> Path:
    return FIGURE_EXPORTS.candidate_path(filename)


def figure_path(filename: str | Path) -> Path:
    return FIGURE_EXPORTS.figure_path(filename)


def reporter_display(reporter: str) -> str:
    return DISPLAY_REPORTER_LABELS.get(reporter, reporter)


def display_text(text: str) -> str:
    output = str(text)
    for key, value in DISPLAY_REPORTER_LABELS.items():
        output = output.replace(key, value)
    return output


def should_offset_time_column(column_name: str) -> bool:
    column = str(column_name)
    if not column.endswith("_hours"):
        return False
    blocked_tokens = ["relative_", "lag_", "rise_", "interval_"]
    return not any(token in column for token in blocked_tokens)


def display_time_hours(values):
    return _display_time_hours(values, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hours(value: float, decimals: int = 1) -> str:
    return _format_display_hours(value, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def format_display_hour_range(start: float, end: float, decimals: int = 0) -> str:
    return _format_display_hour_range(start, end, decimals=decimals, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def set_display_time_axis(ax, axis: str = "x", crowded: bool = False) -> None:
    _set_display_time_axis(ax, axis=axis, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def set_display_time_colorbar(cbar, crowded: bool = True) -> None:
    _set_display_time_colorbar(cbar, crowded=crowded, offset_hours=TIME_DISPLAY_OFFSET_HOURS)


def display_time_df(df: pd.DataFrame) -> pd.DataFrame:
    return _offset_time_hours_df(
        df,
        should_offset_column=should_offset_time_column,
        offset_hours=TIME_DISPLAY_OFFSET_HOURS,
    )

threshold_lookup = {
    row["reporter"]: float(row["threshold_value"])
    for _, row in global_thresholds.iterrows()
}
frame_area = (
    frame_metrics.loc[:, ["position_label", "time_index", "reporter", "organoid_area_px"]]
    .drop_duplicates()
)
if "organoid_area_px" not in population_metrics.columns:
    population_metrics = population_metrics.merge(
        frame_area,
        on=["position_label", "time_index", "reporter"],
        how="left",
    )
for sigma_threshold in [2, 3, 4, 5]:
    fraction_col = f"positive_fraction_sigma{sigma_threshold}"
    mean_col = f"positive_mean_intensity_sigma{sigma_threshold}"
    integrated_col = f"positive_integrated_intensity_sigma{sigma_threshold}"
    if (
        fraction_col in population_metrics.columns
        and mean_col in population_metrics.columns
        and integrated_col not in population_metrics.columns
    ):
        population_metrics[integrated_col] = (
            population_metrics["organoid_area_px"]
            * population_metrics[fraction_col]
            * population_metrics[mean_col]
        )
population_metrics["positive_integrated_intensity_theme2_default"] = np.where(
    population_metrics["reporter"].eq("RFP"),
    population_metrics["positive_integrated_intensity_sigma4"],
    population_metrics["positive_integrated_intensity_sigma3"],
)
population_metrics["positive_mean_intensity_theme2_default_z"] = np.where(
    population_metrics["reporter"].eq("RFP"),
    population_metrics["positive_mean_intensity_sigma4_z"],
    population_metrics["positive_mean_intensity_sigma3_z"],
)
population_metrics["positive_area_theme2_default"] = np.where(
    population_metrics["reporter"].eq("RFP"),
    population_metrics["organoid_area_px"] * population_metrics["positive_fraction_sigma4"],
    population_metrics["organoid_area_px"] * population_metrics["positive_fraction_sigma3"],
)
GLOBAL_NORMALIZATION_CACHE = {}
PLOT_COUNTER = 0


def next_plot_title(title: str) -> str:
    global PLOT_COUNTER
    PLOT_COUNTER += 1
    return f"{PLOT_COUNTER}. {title}"

halfmax_cache_existed = HALFMAX_CACHE_PATH.exists()
halfmax_df = load_or_build_halfmax_cache(
    population_metrics=population_metrics,
    cache_path=HALFMAX_CACHE_PATH,
    metric_names=COMMON_HALFMAX_METRICS,
    smooth_sigma=4.0,
)
halfmax_lookup = build_halfmax_lookup(halfmax_df)

display(
    Markdown(
        f'''
        **Loaded shared cooperativity inputs**

        - population trace rows: `{len(population_metrics):,}`
        - positions in population table: `{population_metrics["position_label"].nunique():,}`
        - shared half-max cache: `{"reused existing" if halfmax_cache_existed else "created new"}`
        - half-max cache path: `{HALFMAX_CACHE_PATH.name}`
        - default thresholded pairing for Theme 2:
          - `RFP`: `{DEFAULT_RFP_SIGMA} sigma`
          - `YFP`: `{DEFAULT_YFP_SIGMA} sigma`
        '''
    )
)


In [ ]:
def build_joint_metric_table(
    metric_name_by_reporter: dict[str, str],
    normalization_mode: str | None = None,
    smooth_sigma: float = 4.0,
    clip_normalized: bool = False,
) -> pd.DataFrame:
    joint_rows = []
    for position_label, position_df in population_metrics.groupby("position_label", sort=True):
        per_reporter_tables = {}
        for reporter in ["RFP", "YFP"]:
            metric_name = metric_name_by_reporter[reporter]
            subset = (
                position_df.loc[
                    position_df["reporter"] == reporter,
                    ["time_hours", metric_name],
                ]
                .sort_values("time_hours")
                .copy()
            )
            raw_values = subset[metric_name].to_numpy(dtype=float)
            if normalization_mode == "per_position":
                display_values = per_position_normalize_trace_for_display(
                    raw_values,
                    early_n=8,
                    smooth_sigma=smooth_sigma,
                    clip_normalized=clip_normalized,
                )
            elif normalization_mode == "global":
                display_values = globally_normalize_trace_for_display(
                    raw_values,
                    metric_name=metric_name,
                    reporter=reporter,
                    early_n=8,
                    smooth_sigma=smooth_sigma,
                    clip_normalized=clip_normalized,
                )
            else:
                display_values = raw_values
            per_reporter_tables[reporter] = (
                subset.assign(display_value=display_values)[["time_hours", "display_value"]]
                .rename(columns={"display_value": f"{reporter.lower()}_value"})
            )

        rfp_subset = per_reporter_tables["RFP"]
        yfp_subset = per_reporter_tables["YFP"]
        joint = (
            rfp_subset.merge(yfp_subset, on="time_hours", how="inner")
            .sort_values("time_hours")
            .dropna()
        )
        if joint.empty:
            continue
        joint["position_label"] = position_label
        joint_rows.append(joint)
    if not joint_rows:
        return pd.DataFrame(columns=["time_hours", "rfp_value", "yfp_value", "position_label"])
    return pd.concat(joint_rows, ignore_index=True)


def bin_joint_trajectory(joint_df: pd.DataFrame, bin_width_hours: float = 4.0) -> pd.DataFrame:
    if joint_df.empty:
        return joint_df.copy()
    binned = joint_df.copy()
    binned["time_bin_center"] = (
        np.floor(binned["time_hours"].to_numpy(dtype=float) / float(bin_width_hours)) * float(bin_width_hours)
        + float(bin_width_hours) / 2.0
    )
    return (
        binned.groupby(["position_label", "time_bin_center"], as_index=False)
        .agg(
            rfp_value=("rfp_value", "mean"),
            yfp_value=("yfp_value", "mean"),
            n_frames=("time_hours", "size"),
        )
        .sort_values(["position_label", "time_bin_center"])
        .reset_index(drop=True)
    )


def build_mean_joint_path(mean_x: pd.DataFrame, mean_y: pd.DataFrame) -> pd.DataFrame:
    return (
        mean_x[["time_hours", "mean"]]
        .rename(columns={"mean": "rfp_mean"})
        .merge(
            mean_y[["time_hours", "mean"]].rename(columns={"mean": "yfp_mean"}),
            on="time_hours",
            how="inner",
        )
        .sort_values("time_hours")
        .reset_index(drop=True)
    )


def build_velocity_segments(joint_df: pd.DataFrame, bin_width_hours: float = 4.0) -> pd.DataFrame:
    binned_df = bin_joint_trajectory(joint_df, bin_width_hours=bin_width_hours)
    rows = []
    for position_label, joint in binned_df.groupby("position_label", sort=True):
        joint = joint.sort_values("time_bin_center")
        if len(joint) < 2:
            continue
        x = joint["rfp_value"].to_numpy(dtype=float)
        y = joint["yfp_value"].to_numpy(dtype=float)
        t = joint["time_bin_center"].to_numpy(dtype=float)
        finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(t)
        x = x[finite]
        y = y[finite]
        t = t[finite]
        if len(t) < 2:
            continue
        for idx in range(len(t) - 1):
            dt = float(t[idx + 1] - t[idx])
            if not np.isfinite(dt) or dt <= 0:
                continue
            rows.append(
                {
                    "position_label": position_label,
                    "time_mid": float(0.5 * (t[idx] + t[idx + 1])),
                    "x_mid": float(0.5 * (x[idx] + x[idx + 1])),
                    "y_mid": float(0.5 * (y[idx] + y[idx + 1])),
                    "u_per_hour": float((x[idx + 1] - x[idx]) / dt),
                    "v_per_hour": float((y[idx + 1] - y[idx]) / dt),
                }
            )
    return pd.DataFrame(rows)


def representative_positions_by_positive_fraction_lag(
    n_positions: int = 12,
    return_details: bool = False,
):
    global_max_time = float(population_metrics["time_hours"].max())
    min_required_final_time = global_max_time - 4.0
    rows = []
    for position_label, _ in population_metrics.groupby("position_label", sort=True):
        rfp_df = (
            population_metrics.loc[
                (population_metrics["position_label"] == position_label)
                & (population_metrics["reporter"] == "RFP"),
                ["time_hours", "positive_fraction_sigma4"],
            ]
            .rename(columns={"positive_fraction_sigma4": "rfp_value"})
        )
        yfp_df = (
            population_metrics.loc[
                (population_metrics["position_label"] == position_label)
                & (population_metrics["reporter"] == "YFP"),
                ["time_hours", "positive_fraction_sigma3"],
            ]
            .rename(columns={"positive_fraction_sigma3": "yfp_value"})
        )
        joint = (
            rfp_df.merge(yfp_df, on="time_hours", how="inner")
            .sort_values("time_hours")
            .reset_index(drop=True)
        )
        if len(joint) < 3:
            continue
        x = joint["rfp_value"].to_numpy(dtype=float)
        y = joint["yfp_value"].to_numpy(dtype=float)
        finite = np.isfinite(x) & np.isfinite(y)
        if finite.sum() < 3:
            continue
        x = x[finite]
        y = y[finite]
        start_rfp = float(x[0])
        start_yfp = float(y[0])
        end_rfp = float(x[-1])
        end_yfp = float(y[-1])
        final_time = float(joint.loc[finite, "time_hours"].to_numpy(dtype=float)[-1])
        rfp_range = float(np.nanmax(x) - np.nanmin(x))
        yfp_range = float(np.nanmax(y) - np.nanmin(y))
        path_length = float(np.sum(np.hypot(np.diff(x), np.diff(y))))
        if (
            start_yfp > 0.18
            or start_rfp > 0.45
            or end_yfp < 0.35
            or final_time < min_required_final_time
        ):
            continue
        dynamic_score = float(
            1.0 * rfp_range
            + 1.8 * yfp_range
            + 0.45 * path_length
            + 0.85 * end_yfp
            + 0.15 * end_rfp
            - 2.1 * start_yfp
            - 1.1 * start_rfp
        )
        rows.append(
            {
                "position_label": position_label,
                "start_rfp": start_rfp,
                "start_yfp": start_yfp,
                "end_rfp": end_rfp,
                "end_yfp": end_yfp,
                "rfp_range": rfp_range,
                "yfp_range": yfp_range,
                "path_length": path_length,
                "dynamic_score": dynamic_score,
                "final_time": final_time,
            }
        )
    score_df = pd.DataFrame(rows).sort_values(
        ["dynamic_score", "end_yfp", "yfp_range", "path_length"],
        ascending=False,
    ).reset_index(drop=True)
    if score_df.empty:
        if return_details:
            return {
                "selected_positions": [],
                "selected_df": pd.DataFrame(),
                "candidate_pool_df": pd.DataFrame(),
                "score_df": pd.DataFrame(),
            }
        return []
    candidate_pool = score_df.head(max(n_positions * 4, n_positions)).copy()
    chosen_df = candidate_pool.sort_values(
        ["start_rfp", "start_yfp", "end_yfp", "dynamic_score"],
        ascending=[True, True, False, False],
    ).reset_index(drop=True)
    selected_df = chosen_df.head(n_positions).copy().reset_index(drop=True)
    selected_df.insert(0, "selection_rank", np.arange(1, len(selected_df) + 1))
    if return_details:
        return {
            "selected_positions": selected_df["position_label"].tolist(),
            "selected_df": selected_df,
            "candidate_pool_df": candidate_pool.reset_index(drop=True),
            "score_df": score_df,
        }
    return selected_df["position_label"].tolist()


def aggregate_display_trace(
    metric_name: str,
    reporter: str,
    normalization_mode: str | None = None,
    clip_normalized: bool = False,
) -> pd.DataFrame:
    subset = population_metrics.loc[population_metrics["reporter"] == reporter].copy()
    if normalization_mode is None:
        summary = aggregate_mean_trace(population_metrics, metric_name, reporter)
        return summary[["time_hours", "mean", "std", "count"]].copy()

    normalized_rows = []
    for position_label, position_df in subset.groupby("position_label", sort=True):
        position_df = position_df.sort_values("time_hours")
        raw_values = position_df[metric_name].to_numpy(dtype=float)
        if normalization_mode == "per_position":
            normalized = per_position_normalize_trace_for_display(
                raw_values,
                early_n=8,
                smooth_sigma=4.0,
                clip_normalized=clip_normalized,
            )
        elif normalization_mode == "global":
            normalized = globally_normalize_trace_for_display(
                raw_values,
                metric_name=metric_name,
                reporter=reporter,
                early_n=8,
                smooth_sigma=4.0,
                clip_normalized=clip_normalized,
            )
        else:
            raise ValueError(f"Unsupported normalization_mode: {normalization_mode}")
        for time_h, value in zip(position_df["time_hours"].to_numpy(dtype=float), normalized):
            if np.isfinite(value):
                normalized_rows.append(
                    {
                        "position_label": position_label,
                        "time_hours": float(time_h),
                        "normalized_value": float(value),
                    }
                )

    normalized_df = pd.DataFrame(normalized_rows)
    if normalized_df.empty:
        return pd.DataFrame(columns=["time_hours", "mean", "std", "count"])
    return (
        normalized_df.groupby("time_hours", as_index=False)
        .agg(
            mean=("normalized_value", "mean"),
            std=("normalized_value", "std"),
            count=("normalized_value", "size"),
        )
        .sort_values("time_hours")
        .reset_index(drop=True)
    )


def render_mean_grid(
    metric_specs: list[dict[str, str]],
    figure_path: Path,
    figure_title: str,
    normalization_mode: str | None = None,
    clip_normalized: bool = False,
    ci_level: float | None = None,
    figure_size: tuple[float, float] | None = None,
    panel_title_fontsize: float = 10.0,
) -> None:
    n_cols = len(metric_specs)
    fig, axes = plt.subplots(
        1,
        n_cols,
        figsize=figure_size if figure_size is not None else (5.0 * n_cols, 4.0),
        sharex=True,
        constrained_layout=True,
    )
    if n_cols == 1:
        axes = np.asarray([axes])
    for ax, spec in zip(axes, metric_specs):
        plotted = []
        spread_style = spec.get("spread_style", "std")
        panel_normalization_mode = spec.get("normalization_mode_override", normalization_mode)
        for reporter in ["RFP", "YFP"]:
            metric_name = spec.get("metric_name_by_reporter", {}).get(reporter, spec.get("metric_name"))
            if spread_style == "iqr" and panel_normalization_mode is None:
                reporter_df = population_metrics.loc[
                    population_metrics["reporter"] == reporter,
                    ["time_hours", metric_name],
                ].copy()
                summary = (
                    reporter_df.groupby("time_hours", as_index=False)
                    .agg(
                        mean=(metric_name, "mean"),
                        std=(metric_name, "std"),
                        q25=(metric_name, lambda x: float(np.nanquantile(np.asarray(x, dtype=float), 0.25))),
                        q75=(metric_name, lambda x: float(np.nanquantile(np.asarray(x, dtype=float), 0.75))),
                        count=(metric_name, "count"),
                    )
                    .sort_values("time_hours")
                    .reset_index(drop=True)
                )
            else:
                summary = aggregate_display_trace(
                    metric_name,
                    reporter,
                    normalization_mode=panel_normalization_mode,
                    clip_normalized=clip_normalized,
                )
            t = summary["time_hours"].to_numpy(dtype=float)
            y = summary["mean"].to_numpy(dtype=float)
            y_std = summary["std"].to_numpy(dtype=float) if "std" in summary.columns else np.full_like(y, np.nan)
            y_count = summary["count"].to_numpy(dtype=float) if "count" in summary.columns else np.full_like(y, np.nan)
            plotted.append(y)
            finite_mean = np.isfinite(y)
            if finite_mean.any():
                plotted.append(y[finite_mean])
            if spread_style == "iqr" and "q25" in summary.columns and "q75" in summary.columns:
                y_lower = summary["q25"].to_numpy(dtype=float)
                y_upper = summary["q75"].to_numpy(dtype=float)
            else:
                y_lower = y - y_std
                y_upper = y + y_std
            finite_band = np.isfinite(t) & np.isfinite(y_lower) & np.isfinite(y_upper)
            if finite_band.any():
                plotted.append(y_lower[finite_band])
                plotted.append(y_upper[finite_band])
                ax.fill_between(
                    t[finite_band],
                    y_lower[finite_band],
                    y_upper[finite_band],
                    color=REPORTER_COLORS[reporter],
                    alpha=REPORTER_FILL_ALPHA,
                    linewidth=0.0,
                )
            if ci_level is not None:
                finite_ci = np.isfinite(t) & np.isfinite(y) & np.isfinite(y_std) & np.isfinite(y_count) & (y_count > 1)
                if finite_ci.any():
                    sem = y_std[finite_ci] / np.sqrt(y_count[finite_ci])
                    t_crit = stats.t.ppf(0.5 + 0.5 * float(ci_level), df=np.maximum(y_count[finite_ci] - 1.0, 1.0))
                    ci_halfwidth = t_crit * sem
                    plotted.append((y[finite_ci] - ci_halfwidth))
                    plotted.append((y[finite_ci] + ci_halfwidth))
                    ax.plot(
                        t[finite_ci],
                        y[finite_ci] - ci_halfwidth,
                        color=REPORTER_COLORS[reporter],
                        linewidth=1.1,
                        linestyle="--",
                        alpha=0.9,
                    )
                    ax.plot(
                        t[finite_ci],
                        y[finite_ci] + ci_halfwidth,
                        color=REPORTER_COLORS[reporter],
                        linewidth=1.1,
                        linestyle="--",
                        alpha=0.9,
                    )
            ax.plot(
                t,
                y,
                color=REPORTER_COLORS[reporter],
                linewidth=2.6,
                label=reporter_display(reporter),
            )
        ax.set_title(next_plot_title(spec["title"]), fontsize=panel_title_fontsize)
        ax.set_xlabel("Time (hours)")
        set_display_time_axis(ax, "x")
        ax.set_ylabel(spec["ylabel"])
        ax.grid(alpha=0.18)
        if spec.get("draw_zero_line", False):
            ax.axhline(0.0, color="0.45", linewidth=1.45, linestyle="--", alpha=0.9, zorder=0)
        if spec.get("y_limits_override") is not None:
            ax.set_ylim(*spec["y_limits_override"])
        elif spec.get("y_limits_quantiles") is not None:
            pooled = np.concatenate(
                [np.asarray(arr, dtype=float)[np.isfinite(arr)] for arr in plotted if np.asarray(arr, dtype=float).size and np.isfinite(arr).any()]
            )
            lower_q, upper_q = spec["y_limits_quantiles"]
            lower = float(np.nanquantile(pooled, lower_q))
            upper = float(np.nanquantile(pooled, upper_q))
            if spec.get("include_zero", panel_normalization_mode is None):
                lower = min(lower, 0.0)
                upper = max(upper, 0.0)
            span = upper - lower
            if not np.isfinite(span) or span <= 0:
                span = max(abs(upper - lower), 1.0)
            pad = max(1e-6, float(spec.get("ylim_padding_fraction", 0.10)) * span)
            ax.set_ylim(lower - pad, upper + pad)
        else:
            ax.set_ylim(
                focus_ylim_from_arrays(
                    plotted,
                    include_zero=spec.get("include_zero", panel_normalization_mode is None),
                    padding_fraction=float(spec.get("ylim_padding_fraction", 0.10)),
                )
            )
        spread_label = "IQR ribbon (25-75%)" if spread_style == "iqr" else "SD ribbon"
        legend_handles = [
            Line2D([0], [0], color=REPORTER_COLORS["RFP"], linewidth=2.6, label=reporter_display("RFP")),
            Line2D([0], [0], color=REPORTER_COLORS["YFP"], linewidth=2.6, label=reporter_display("YFP")),
            Patch(facecolor="0.6", edgecolor="none", alpha=0.25, label=spread_label),
        ]
        if ci_level is not None:
            legend_handles.append(
                Line2D([0], [0], color="0.35", linewidth=1.1, linestyle="--", label=f"{int(round(ci_level * 100))}% CI on mean")
            )
        ax.legend(handles=legend_handles, loc=spec.get("legend_loc", "upper left"), frameon=False)
    fig.suptitle(figure_title, fontsize=10.8)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def global_normalization_params(
    metric_name: str,
    reporter: str,
    early_n: int = 8,
    smooth_sigma: float = 4.0,
) -> tuple[float, float]:
    key = (metric_name, reporter, int(early_n), float(smooth_sigma))
    if key in GLOBAL_NORMALIZATION_CACHE:
        return GLOBAL_NORMALIZATION_CACHE[key]

    summary = aggregate_mean_trace(population_metrics, metric_name, reporter)
    smoothed_mean = gaussian_smooth(summary["mean"].to_numpy(dtype=float), smooth_sigma)
    finite = smoothed_mean[np.isfinite(smoothed_mean)]
    if finite.size < max(early_n, 4):
        params = (float("nan"), float("nan"))
    else:
        baseline = float(np.nanmedian(smoothed_mean[:early_n]))
        peak = float(np.nanmax(smoothed_mean))
        params = (baseline, peak)
    GLOBAL_NORMALIZATION_CACHE[key] = params
    return params


def per_position_normalization_params(
    values: np.ndarray,
    early_n: int = 8,
    smooth_sigma: float = 4.0,
) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    smoothed = gaussian_smooth(arr, smooth_sigma)
    finite = smoothed[np.isfinite(smoothed)]
    if finite.size < max(early_n, 4):
        return (float("nan"), float("nan"))
    baseline = float(np.nanmedian(smoothed[:early_n]))
    peak = float(np.nanmax(smoothed))
    return (baseline, peak)


def per_position_normalize_trace_for_display(
    values: np.ndarray,
    early_n: int = 8,
    smooth_sigma: float = 4.0,
    clip_normalized: bool = False,
) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    baseline, peak = per_position_normalization_params(
        arr,
        early_n=early_n,
        smooth_sigma=smooth_sigma,
    )
    amplitude = peak - baseline
    if not np.isfinite(amplitude) or amplitude <= 0:
        normalized = np.full_like(arr, np.nan, dtype=float)
    else:
        normalized = (arr - baseline) / amplitude
    if clip_normalized:
        normalized = np.clip(normalized, 0.0, 1.0)
    return normalized


def globally_normalize_trace_for_display(
    values: np.ndarray,
    metric_name: str,
    reporter: str,
    early_n: int = 8,
    smooth_sigma: float = 4.0,
    clip_normalized: bool = False,
) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    baseline, peak = global_normalization_params(
        metric_name,
        reporter,
        early_n=early_n,
        smooth_sigma=smooth_sigma,
    )
    amplitude = peak - baseline
    if not np.isfinite(amplitude) or amplitude <= 0:
        normalized = np.full_like(arr, np.nan, dtype=float)
    else:
        normalized = (arr - baseline) / amplitude
    if clip_normalized:
        normalized = np.clip(normalized, 0.0, 1.0)
    return normalized


def globally_normalize_trace(
    values: np.ndarray,
    metric_name: str,
    reporter: str,
    early_n: int = 8,
    smooth_sigma: float = 4.0,
    clip_normalized: bool = False,
) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    smoothed = gaussian_smooth(arr, smooth_sigma)
    baseline, peak = global_normalization_params(
        metric_name,
        reporter,
        early_n=early_n,
        smooth_sigma=smooth_sigma,
    )
    amplitude = peak - baseline
    if not np.isfinite(amplitude) or amplitude <= 0:
        normalized = np.full_like(smoothed, np.nan, dtype=float)
    else:
        normalized = (smoothed - baseline) / amplitude
    if clip_normalized:
        normalized = np.clip(normalized, 0.0, 1.0)
    return normalized


def derivative_gaussian_smooth(
    values: np.ndarray,
    sigma_frames: float,
) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return arr.copy()
    sigma = float(sigma_frames)
    if not np.isfinite(sigma) or sigma <= 0:
        return arr.copy()

    radius = max(1, int(np.ceil(3.0 * sigma)))
    x = np.arange(-radius, radius + 1, dtype=float)
    kernel = np.exp(-0.5 * (x / sigma) ** 2)

    finite_mask = np.isfinite(arr)
    if finite_mask.sum() == 0:
        return np.full_like(arr, np.nan, dtype=float)

    valid_index = np.flatnonzero(finite_mask)
    valid_values = arr[finite_mask]
    filled = np.interp(np.arange(arr.size), valid_index, valid_values)
    smoothed = np.full_like(arr, np.nan, dtype=float)

    for idx in range(arr.size):
        if not finite_mask[idx]:
            continue
        left = max(0, idx - radius)
        right = min(arr.size, idx + radius + 1)
        kernel_left = radius - (idx - left)
        kernel_right = radius + (right - idx)
        local_kernel = kernel[kernel_left:kernel_right]
        weight_sum = float(np.sum(local_kernel))
        if weight_sum <= 0:
            continue
        smoothed[idx] = float(np.dot(filled[left:right], local_kernel) / weight_sum)

    return smoothed


def halfmax_time_for_mode(
    metric_name: str,
    position_label: str,
    reporter: str,
    normalization_mode: str = "per_position",
    clip_normalized: bool = False,
) -> float:
    subset = population_metrics.loc[
        (population_metrics["position_label"] == position_label)
        & (population_metrics["reporter"] == reporter)
    ].sort_values("time_hours")
    values = subset[metric_name].to_numpy(dtype=float)
    time_hours = subset["time_hours"].to_numpy(dtype=float)
    if values.size < 12:
        return float("nan")
    if normalization_mode == "per_position":
        normalized = normalize_trace(values, early_n=8, smooth_sigma=4.0)
        if clip_normalized:
            normalized = np.clip(normalized, 0.0, 1.0)
    elif normalization_mode == "global":
        normalized = globally_normalize_trace(
            values,
            metric_name=metric_name,
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=clip_normalized,
        )
    else:
        raise ValueError(f"Unsupported normalization_mode: {normalization_mode}")
    return first_crossing_time(time_hours, normalized, 0.5)


## Theme 2: RFP Comes Before YFP


### Supplementary Figures


#### Raw Population Views In Clock Time

These plots are the most literal summaries of the current `05` measurements.

They are intentionally shown before any alignment so that the reader can see what the unaligned population means do and do not reveal.


##### Positive-Fraction Trace Summaries

This section keeps the thresholded positive-fraction timecourse separate so we can iterate on it independently if needed.


In [ ]:
positive_fraction_trace_path = figure_path("05c_positive_fraction_trace_summary.png")

fig, ax = plt.subplots(figsize=(6.2, 4.3), constrained_layout=True)
plotted = []
for reporter, metric_name in [("RFP", "positive_fraction_sigma4"), ("YFP", "positive_fraction_sigma3")]:
    reporter_df = population_metrics.loc[
        population_metrics["reporter"] == reporter,
        ["time_hours", metric_name],
    ].copy()
    summary = (
        reporter_df.groupby("time_hours", as_index=False)
        .agg(
            mean=(metric_name, "mean"),
            q25=(metric_name, lambda x: float(np.nanquantile(np.asarray(x, dtype=float), 0.25))),
            q75=(metric_name, lambda x: float(np.nanquantile(np.asarray(x, dtype=float), 0.75))),
            count=(metric_name, "count"),
        )
        .sort_values("time_hours")
        .reset_index(drop=True)
    )
    y = summary["mean"].to_numpy(dtype=float)
    y_lower = summary["q25"].to_numpy(dtype=float)
    y_upper = summary["q75"].to_numpy(dtype=float)
    time_values = summary["time_hours"].to_numpy(dtype=float)
    finite_band = np.isfinite(time_values) & np.isfinite(y_lower) & np.isfinite(y_upper)
    if finite_band.any():
        plotted.append(y_lower[finite_band])
        plotted.append(y_upper[finite_band])
        ax.fill_between(
            time_values[finite_band],
            y_lower[finite_band],
            y_upper[finite_band],
            color=REPORTER_COLORS[reporter],
            alpha=REPORTER_FILL_ALPHA,
            linewidth=0.0,
        )
    finite_mean = np.isfinite(y)
    if finite_mean.any():
        plotted.append(y[finite_mean])
    ax.plot(
        summary["time_hours"],
        y,
        color=REPORTER_COLORS[reporter],
        linewidth=2.7,
        label=reporter,
    )

ax.set_title(next_plot_title("Thresholded reporter-positive fraction"), fontsize=10.0)
ax.set_xlabel("Time (hours)")
set_display_time_axis(ax, "x")
ax.set_ylabel("Mean positive fraction")
ax.set_ylim(
    focus_ylim_from_arrays(
        plotted,
        include_zero=True,
        padding_fraction=0.10,
    )
)
ax.grid(alpha=0.18)
legend_handles = [
    Line2D([0], [0], color=REPORTER_COLORS["RFP"], linewidth=2.7, label=reporter_display("RFP")),
    Line2D([0], [0], color=REPORTER_COLORS["YFP"], linewidth=2.7, label=reporter_display("YFP")),
    Patch(facecolor="0.6", edgecolor="none", alpha=0.25, label="IQR ribbon (25-75%)"),
]
ax.legend(handles=legend_handles, loc="upper left", frameon=False)

fig.suptitle("Positive-fraction population means in clock time", fontsize=11.6)
fig.savefig(positive_fraction_trace_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", positive_fraction_trace_path)


##### Intensity Summaries


In [ ]:
absolute_threshold_free_specs = [
    {
        "metric_name": "organoid_mean_intensity_z",
        "title": "Whole-cyst mean intensity",
        "ylabel": "mean sigma above pooled null",
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "title": "Brightest 10% mean intensity",
        "ylabel": "mean sigma above pooled null",
    },
]
normalized_threshold_free_specs = [
    {
        "metric_name": "organoid_mean_intensity_z",
        "title": "Whole-cyst mean intensity",
        "ylabel": "Mean normalized progression",
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "title": "Brightest 10% mean intensity",
        "ylabel": "Mean normalized progression",
    },
]
global_normalized_threshold_free_specs = [
    {
        "metric_name": "organoid_mean_intensity_z",
        "title": "Whole-cyst mean intensity",
        "ylabel": "Mean normalized progression",
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "title": "Brightest 10% mean intensity",
        "ylabel": "Mean normalized progression",
    },
]

render_mean_grid(
    absolute_threshold_free_specs,
    figure_path("05c_absolute_time_intensity_summaries.png"),
    "Absolute-time population means: whole-cyst and brightest-decile intensity",
    normalization_mode=None,
    ci_level=0.95,
)


In [ ]:
integrated_intensity_scale_col = "positive_integrated_intensity_theme2_default_intensity_scaled"
integrated_intensity_meantrace_normalized_col = "positive_integrated_intensity_theme2_default_meantrace_normalized"
if integrated_intensity_scale_col not in population_metrics.columns:
    scaled_values = np.full(len(population_metrics), np.nan, dtype=float)
    integrated_intensity_scale_metric_by_reporter = {
        "RFP": "positive_mean_intensity_sigma4",
        "YFP": "positive_mean_intensity_sigma3",
    }
    for reporter, mean_metric in integrated_intensity_scale_metric_by_reporter.items():
        baseline, peak = global_normalization_params(
            mean_metric,
            reporter,
            early_n=8,
            smooth_sigma=4.0,
        )
        intensity_scale = peak - baseline
        reporter_mask = population_metrics["reporter"].eq(reporter).to_numpy()
        if np.isfinite(intensity_scale) and intensity_scale > 0:
            scaled_values[reporter_mask] = (
                population_metrics.loc[reporter_mask, "positive_integrated_intensity_theme2_default"].to_numpy(dtype=float)
                / float(intensity_scale)
            )
    population_metrics[integrated_intensity_scale_col] = scaled_values

if integrated_intensity_meantrace_normalized_col not in population_metrics.columns:
    normalized_values = np.full(len(population_metrics), np.nan, dtype=float)
    for reporter in ["RFP", "YFP"]:
        reporter_mask = population_metrics["reporter"].eq(reporter).to_numpy()
        summary = aggregate_mean_trace(
            population_metrics,
            "positive_integrated_intensity_theme2_default",
            reporter,
        )
        mean_values = summary["mean"].to_numpy(dtype=float)
        finite = mean_values[np.isfinite(mean_values)]
        if finite.size < 2:
            continue
        baseline = float(np.nanmin(finite))
        peak = float(np.nanmax(finite))
        amplitude = peak - baseline
        if np.isfinite(amplitude) and amplitude > 0:
            normalized_values[reporter_mask] = (
                population_metrics.loc[reporter_mask, "positive_integrated_intensity_theme2_default"].to_numpy(dtype=float)
                - baseline
            ) / float(amplitude)
    population_metrics[integrated_intensity_meantrace_normalized_col] = normalized_values

global_normalized_intensity_specs = [
    *global_normalized_threshold_free_specs,
]

render_mean_grid(
    global_normalized_intensity_specs,
    figure_path("05c_global_normalized_intensity_progression.png"),
    "Population-wide normalized progression: threshold-free reporter summaries",
    normalization_mode="global",
    clip_normalized=False,
    ci_level=0.95,
)


In [ ]:
per_cyst_normalized_threshold_free_specs = [
    {
        "metric_name": "organoid_mean_intensity_z",
        "title": "Whole-cyst mean intensity*",
        "ylabel": "Mean normalized progression",
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "title": "Brightest 10% mean intensity",
        "ylabel": "Mean normalized progression",
    },
]

render_mean_grid(
    per_cyst_normalized_threshold_free_specs,
    figure_path("05c_per_cyst_normalized_threshold_free_progression.png"),
    "Per-cyst normalized progression: threshold-free reporter summaries",
    normalization_mode="per_position",
    clip_normalized=False,
    ci_level=0.95,
)


*Interpretation note for the whole-cyst mean-intensity panel:* the mid-course dip in the broader `YFP` intensity summary is consistent with cyst/tissue growth preceding expansion of the `YFP` domain. The adjacent brightest-10% summary does not show the same dip as strongly, which argues against a simple collapse of the brightest `YFP` signal and instead supports the idea that the brightest `YFP` pixels persist while the surrounding cyst context expands.


##### Thresholded Positive-Region Mean-Intensity Summaries

These panels ask a slightly different question from the threshold-free summaries above.

Instead of averaging over the whole cyst, they summarize how bright the already-positive reporter domain is under the default Theme 2 thresholds:

- `RFP`: `4 sigma`
- `YFP`: `3 sigma`


In [ ]:
positive_region_mean_intensity_specs = [
    {
        "metric_name": "positive_mean_intensity_theme2_default_z",
        "title": "Mean intensity within reporter-positive area",
        "ylabel": "mean sigma above pooled null",
        "normalization_mode_override": None,
    },
    {
        "metric_name": "positive_mean_intensity_theme2_default_z",
        "title": "Mean intensity within reporter-positive area (population-wide normalized)",
        "ylabel": "Mean normalized progression",
        "normalization_mode_override": "global",
    },
    {
        "metric_name": "positive_mean_intensity_theme2_default_z",
        "title": "Mean intensity within reporter-positive area (per-cyst normalized)",
        "ylabel": "Mean normalized progression",
        "normalization_mode_override": "per_position",
        "y_limits_quantiles": (0.10, 0.90),
        "ylim_padding_fraction": 0.05,
    },
]

render_mean_grid(
    positive_region_mean_intensity_specs,
    figure_path("05c_positive_region_mean_intensity_summaries.png"),
    "Thresholded positive-region mean-intensity summaries",
    normalization_mode=None,
    clip_normalized=False,
    ci_level=0.95,
    figure_size=(16.0, 4.4),
    panel_title_fontsize=9.2,
)


##### Intensity Derivatives

These derivative panels are based on the same population-wide normalized intensity metrics shown above.

To estimate derivatives, each position-level intensity trace is first put on the shared population-wide normalization scale for its reporter, then Gaussian-smoothed (`sigma = 16` frames) with a `05c`-specific boundary-aware smoother that uses only in-range values near the edges (no edge padding) before taking `d/dt`. The plotted derivative summaries are then aggregated across positions in clock time.


In [ ]:
DERIVATIVE_SMOOTH_SIGMA = 16.0
derivative_metric_map = {
    "organoid_mean_intensity_z": "organoid_mean_intensity_z_global_dt",
    "brightest_decile_mean_intensity_z": "brightest_decile_mean_intensity_z_global_dt",
    "positive_mean_intensity_theme2_default_z": "positive_mean_intensity_theme2_default_z_global_dt",
}

for metric_name, derivative_col in derivative_metric_map.items():
    derivative_series = pd.Series(np.nan, index=population_metrics.index, dtype=float)
    for (reporter, _), group_idx in population_metrics.groupby(["reporter", "position_label"], sort=True).groups.items():
        ordered = (
            population_metrics.loc[list(group_idx), ["time_hours", metric_name]]
            .sort_values("time_hours")
        )
        time_hours = ordered["time_hours"].to_numpy(dtype=float)
        raw_values = ordered[metric_name].to_numpy(dtype=float)
        normalized_values = globally_normalize_trace_for_display(
            raw_values,
            metric_name=metric_name,
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        if finite.sum() < 2:
            continue
        smoothed_values = derivative_gaussian_smooth(normalized_values, DERIVATIVE_SMOOTH_SIGMA)
        derivative_values = np.full_like(normalized_values, np.nan, dtype=float)
        derivative_values[finite] = local_gradient(smoothed_values[finite], time_hours[finite])
        derivative_series.loc[ordered.index] = derivative_values
    population_metrics[derivative_col] = derivative_series.to_numpy(dtype=float)

derivative_specs = [
    {
        "metric_name": "organoid_mean_intensity_z_global_dt",
        "title": "Whole-cyst mean intensity derivative",
        "ylabel": "Mean d/dt (population-normalized progression per hour)",
        "draw_zero_line": True,
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z_global_dt",
        "title": "Brightest 10% mean intensity derivative",
        "ylabel": "Mean d/dt (population-normalized progression per hour)",
        "draw_zero_line": True,
    },
    {
        "metric_name": "positive_mean_intensity_theme2_default_z_global_dt",
        "title": "Mean intensity within reporter-positive area derivative",
        "ylabel": "Mean d/dt (population-normalized progression per hour)",
        "draw_zero_line": True,
    },
]

render_mean_grid(
    derivative_specs,
    figure_path("05c_threshold_free_intensity_derivatives.png"),
    "Intensity derivatives in clock time",
    normalization_mode=None,
    ci_level=0.95,
    figure_size=(18.0, 4.9),
    panel_title_fontsize=9.4,
)


##### Aggregate-Trace Derivative Alternatives

These alternatives use the original aggregate mean traces for the same population-normalized metrics above.

For each reporter, the aggregate mean trace is first computed in clock time, then Gaussian-smoothed with the same `05c`-specific boundary-aware smoother (`sigma = 16` frames, no edge padding), and only then differentiated. Unlike the panels above, these views do not average derivatives across positions.


In [ ]:
aggregate_derivative_specs = [
    {
        "metric_name": "organoid_mean_intensity_z",
        "title": "Whole-cyst mean intensity | derivative of smoothed aggregate mean",
        "ylabel": "d/dt of aggregate mean (population-normalized progression per hour)",
    },
    {
        "metric_name": "brightest_decile_mean_intensity_z",
        "title": "Brightest 10% mean intensity | derivative of smoothed aggregate mean",
        "ylabel": "d/dt of aggregate mean (population-normalized progression per hour)",
    },
    {
        "metric_name": "positive_mean_intensity_theme2_default_z",
        "title": "Mean intensity within reporter-positive area | derivative of smoothed aggregate mean",
        "ylabel": "d/dt of aggregate mean (population-normalized progression per hour)",
    },
]

fig, axes = plt.subplots(1, len(aggregate_derivative_specs), figsize=(18.0, 5.1), sharex=True)
if len(aggregate_derivative_specs) == 1:
    axes = np.asarray([axes])
aggregate_bootstrap_rng = np.random.default_rng(20260329)
aggregate_bootstrap_reps = 400
legend_handles = [
    Line2D([0], [0], color=REPORTER_COLORS["RFP"], linewidth=2.6),
    Line2D([0], [0], color=REPORTER_COLORS["YFP"], linewidth=2.6),
    Patch(facecolor="0.75", edgecolor="none", alpha=0.22),
]
legend_labels = [reporter_display("RFP"), reporter_display("YFP"), "95% bootstrap CI"]

for ax, spec in zip(axes, aggregate_derivative_specs):
    plotted = []
    for reporter in ["RFP", "YFP"]:
        reporter_df = population_metrics.loc[population_metrics["reporter"] == reporter].copy()
        trace_rows = []
        for position_label, position_df in reporter_df.groupby("position_label", sort=True):
            position_df = position_df.sort_values("time_hours")
            time_hours = position_df["time_hours"].to_numpy(dtype=float)
            raw_values = position_df[spec["metric_name"]].to_numpy(dtype=float)
            normalized_values = globally_normalize_trace_for_display(
                raw_values,
                metric_name=spec["metric_name"],
                reporter=reporter,
                early_n=8,
                smooth_sigma=4.0,
                clip_normalized=False,
            )
            finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
            for time_h, value in zip(time_hours[finite], normalized_values[finite]):
                trace_rows.append(
                    {
                        "position_label": position_label,
                        "time_hours": float(time_h),
                        "normalized_value": float(value),
                    }
                )
        if not trace_rows:
            continue
        trace_df = pd.DataFrame(trace_rows)
        matrix_df = (
            trace_df.pivot(index="position_label", columns="time_hours", values="normalized_value")
            .sort_index(axis=1)
        )
        time_hours = matrix_df.columns.to_numpy(dtype=float)
        matrix = matrix_df.to_numpy(dtype=float)
        if matrix.shape[0] == 0 or matrix.shape[1] < 2:
            continue

        mean_values = np.nanmean(matrix, axis=0)
        smoothed_mean = derivative_gaussian_smooth(mean_values, DERIVATIVE_SMOOTH_SIGMA)
        finite_smoothed = np.isfinite(time_hours) & np.isfinite(smoothed_mean)
        if finite_smoothed.sum() < 2:
            continue
        central_derivative = np.full_like(smoothed_mean, np.nan, dtype=float)
        central_derivative[finite_smoothed] = local_gradient(
            smoothed_mean[finite_smoothed],
            time_hours[finite_smoothed],
        )

        bootstrap_curves = []
        for _ in range(aggregate_bootstrap_reps):
            sample_idx = aggregate_bootstrap_rng.integers(0, matrix.shape[0], size=matrix.shape[0])
            sampled_mean = np.nanmean(matrix[sample_idx], axis=0)
            sampled_smoothed = derivative_gaussian_smooth(sampled_mean, DERIVATIVE_SMOOTH_SIGMA)
            finite_sampled = np.isfinite(time_hours) & np.isfinite(sampled_smoothed)
            if finite_sampled.sum() < 2 or not np.array_equal(finite_sampled, finite_smoothed):
                continue
            sampled_derivative = np.full_like(sampled_smoothed, np.nan, dtype=float)
            sampled_derivative[finite_sampled] = local_gradient(
                sampled_smoothed[finite_sampled],
                time_hours[finite_sampled],
            )
            bootstrap_curves.append(sampled_derivative[finite_smoothed])

        if bootstrap_curves:
            bootstrap_curves = np.vstack(bootstrap_curves)
            ci_low = np.nanquantile(bootstrap_curves, 0.025, axis=0)
            ci_high = np.nanquantile(bootstrap_curves, 0.975, axis=0)
        else:
            ci_low = np.full(finite_smoothed.sum(), np.nan, dtype=float)
            ci_high = np.full(finite_smoothed.sum(), np.nan, dtype=float)

        finite_time = time_hours[finite_smoothed]
        central_values = central_derivative[finite_smoothed]
        plotted.extend([central_values, ci_low, ci_high])
        ax.fill_between(
            finite_time,
            ci_low,
            ci_high,
            color=REPORTER_COLORS[reporter],
            alpha=0.16,
            linewidth=0.0,
            zorder=1,
        )
        ax.plot(
            finite_time,
            central_values,
            color=REPORTER_COLORS[reporter],
            linewidth=2.6,
            zorder=3,
        )

    ax.axhline(0.0, color="0.45", linewidth=1.45, linestyle="--", alpha=0.9, zorder=0)
    ax.set_title(next_plot_title(spec["title"]), fontsize=9.4)
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel(spec["ylabel"])
    ax.grid(alpha=0.18)
    ax.set_ylim(
        focus_ylim_from_arrays(
            plotted,
            include_zero=True,
            padding_fraction=0.10,
        )
    )
fig.legend(
    legend_handles,
    legend_labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.93),
    ncol=3,
    frameon=False,
    fontsize=8.7,
    handlelength=2.1,
    columnspacing=1.3,
)
fig.suptitle("Aggregate-trace derivative alternatives in clock time", fontsize=11.6, y=0.98)
fig.subplots_adjust(left=0.06, right=0.99, bottom=0.12, top=0.77, wspace=0.26)
fig.savefig(figure_path("05c_threshold_free_intensity_derivatives_aggregate_mean.png"), dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", figure_path("05c_threshold_free_intensity_derivatives_aggregate_mean.png"))


##### Bootstrap-Based Sigma Selection

This table treats positions as the independent unit and summarizes three quantities for each smoothing choice:

- `roughness_rms_per_hour2`: RMS time-derivative of the derivative curve, used as a compact wiggliness measure
- `bootstrap_mean_pointwise_iqr`: average pointwise bootstrap IQR width for the mean derivative curve across positions
- `rmse_vs_derivative_of_mean`: RMS difference between the current main method (mean of per-position derivatives) and the alternative method (derivative of the aggregate mean trace)

The intent is to identify the smallest sigma that yields a reasonably stable derivative without obvious over-smoothing.


In [ ]:
BOOTSTRAP_REPLICATES = 200
bootstrap_sigma_candidates = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 16.0, 20.0, 24.0, 30.0]
bootstrap_rng = np.random.default_rng(20260327)
derivative_bootstrap_specs = [
    {"metric_name": "organoid_mean_intensity_z", "metric_title": "Whole-cyst mean intensity", "reporter": "RFP"},
    {"metric_name": "organoid_mean_intensity_z", "metric_title": "Whole-cyst mean intensity", "reporter": "YFP"},
    {"metric_name": "brightest_decile_mean_intensity_z", "metric_title": "Brightest 10% mean intensity", "reporter": "RFP"},
    {"metric_name": "brightest_decile_mean_intensity_z", "metric_title": "Brightest 10% mean intensity", "reporter": "YFP"},
]

def build_population_normalized_trace_matrix(metric_name: str, reporter: str) -> tuple[np.ndarray, np.ndarray]:
    rows = []
    reporter_df = population_metrics.loc[population_metrics["reporter"] == reporter].copy()
    for position_label, position_df in reporter_df.groupby("position_label", sort=True):
        position_df = position_df.sort_values("time_hours")
        time_hours = position_df["time_hours"].to_numpy(dtype=float)
        raw_values = position_df[metric_name].to_numpy(dtype=float)
        normalized_values = globally_normalize_trace_for_display(
            raw_values,
            metric_name=metric_name,
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        for time_h, value in zip(time_hours[finite], normalized_values[finite]):
            rows.append(
                {
                    "position_label": position_label,
                    "time_hours": float(time_h),
                    "normalized_value": float(value),
                }
            )
    normalized_trace_df = pd.DataFrame(rows)
    if normalized_trace_df.empty:
        return np.empty((0, 0), dtype=float), np.empty((0,), dtype=float)
    pivot = (
        normalized_trace_df.pivot(index="position_label", columns="time_hours", values="normalized_value")
        .sort_index(axis=0)
        .sort_index(axis=1)
    )
    return pivot.to_numpy(dtype=float), pivot.columns.to_numpy(dtype=float)

def compute_derivative_matrix(matrix: np.ndarray, time_hours: np.ndarray, sigma: float) -> np.ndarray:
    if matrix.size == 0:
        return np.empty_like(matrix)
    derivative_matrix = np.full_like(matrix, np.nan, dtype=float)
    for row_idx in range(matrix.shape[0]):
        values = matrix[row_idx].copy()
        finite = np.isfinite(time_hours) & np.isfinite(values)
        if finite.sum() < 2:
            continue
        source = derivative_gaussian_smooth(values, sigma) if sigma > 0 else values
        finite_source = np.isfinite(time_hours) & np.isfinite(source)
        if finite_source.sum() < 2:
            continue
        derivative_matrix[row_idx, finite_source] = local_gradient(source[finite_source], time_hours[finite_source])
    return derivative_matrix

def compute_derivative_of_mean_curve(matrix: np.ndarray, time_hours: np.ndarray, sigma: float) -> np.ndarray:
    if matrix.size == 0:
        return np.empty((0,), dtype=float)
    mean_trace = np.nanmean(matrix, axis=0)
    source = derivative_gaussian_smooth(mean_trace, sigma) if sigma > 0 else mean_trace
    finite = np.isfinite(time_hours) & np.isfinite(source)
    curve = np.full_like(source, np.nan, dtype=float)
    if finite.sum() >= 2:
        curve[finite] = local_gradient(source[finite], time_hours[finite])
    return curve

def derivative_curve_roughness(curve: np.ndarray, time_hours: np.ndarray) -> float:
    finite = np.isfinite(time_hours) & np.isfinite(curve)
    if finite.sum() < 3:
        return float("nan")
    derivative_of_derivative = local_gradient(curve[finite], time_hours[finite])
    finite_second = np.isfinite(derivative_of_derivative)
    if finite_second.sum() == 0:
        return float("nan")
    return float(np.sqrt(np.nanmean(derivative_of_derivative[finite_second] ** 2)))

def mean_pointwise_bootstrap_iqr(bootstrap_curves: np.ndarray) -> float:
    if bootstrap_curves.size == 0:
        return float("nan")
    q25 = np.nanquantile(bootstrap_curves, 0.25, axis=0)
    q75 = np.nanquantile(bootstrap_curves, 0.75, axis=0)
    widths = q75 - q25
    finite = np.isfinite(widths)
    if finite.sum() == 0:
        return float("nan")
    return float(np.nanmean(widths[finite]))

bootstrap_summary_rows = []
for spec in derivative_bootstrap_specs:
    matrix, time_hours = build_population_normalized_trace_matrix(spec["metric_name"], spec["reporter"])
    if matrix.size == 0:
        continue
    n_positions = matrix.shape[0]
    for sigma in bootstrap_sigma_candidates:
        derivative_matrix = compute_derivative_matrix(matrix, time_hours, sigma)
        central_curve = np.nanmean(derivative_matrix, axis=0)
        alternative_curve = compute_derivative_of_mean_curve(matrix, time_hours, sigma)

        bootstrap_curves = []
        for _ in range(BOOTSTRAP_REPLICATES):
            sample_idx = bootstrap_rng.integers(0, n_positions, size=n_positions)
            sampled_curve = np.nanmean(derivative_matrix[sample_idx], axis=0)
            bootstrap_curves.append(sampled_curve)
        bootstrap_curves = np.vstack(bootstrap_curves)

        finite_rmse = np.isfinite(central_curve) & np.isfinite(alternative_curve)
        rmse = float("nan")
        if finite_rmse.sum() > 0:
            rmse = float(np.sqrt(np.nanmean((central_curve[finite_rmse] - alternative_curve[finite_rmse]) ** 2)))

        bootstrap_summary_rows.append(
            {
                "metric_title": spec["metric_title"],
                "reporter": spec["reporter"],
                "sigma_frames": int(sigma),
                "n_positions": int(n_positions),
                "roughness_rms_per_hour2": derivative_curve_roughness(central_curve, time_hours),
                "bootstrap_mean_pointwise_iqr": mean_pointwise_bootstrap_iqr(bootstrap_curves),
                "rmse_vs_derivative_of_mean": rmse,
            }
        )

derivative_bootstrap_summary_df = (
    pd.DataFrame(bootstrap_summary_rows)
    .sort_values(["metric_title", "reporter", "sigma_frames"])
    .reset_index(drop=True)
)
derivative_bootstrap_summary_path = TABLE_DIR / "05c_derivative_sigma_bootstrap_summary.tsv"
derivative_bootstrap_summary_df.to_csv(derivative_bootstrap_summary_path, sep="\t", index=False)
display(display_time_df(derivative_bootstrap_summary_df).style.hide(axis="index").format(
    {
        "roughness_rms_per_hour2": "{:.4f}",
        "bootstrap_mean_pointwise_iqr": "{:.4f}",
        "rmse_vs_derivative_of_mean": "{:.4f}",
    }
))
print("Wrote table:", derivative_bootstrap_summary_path)


##### Derivative Debugging

These debugging figures remove the ribbons and compare derivative behavior across several smoothing choices.

All smoothing in this section uses the same `05c`-specific boundary-aware Gaussian smoother with no edge padding.

There are three views:

- mean of per-position derivatives across smoothing parameters
- derivative of the aggregate mean trace across smoothing parameters
- direct comparison of those two definitions for `sigma = 16` frames


In [ ]:
derivative_debug_sigmas = [0.0, 2.0, 12.0, 16.0]
derivative_debug_colors = {
    0.0: "#111111",
    2.0: "#4c78a8",
    12.0: "#59a14f",
    16.0: "#f28e2b",
}
derivative_debug_specs = [
    {"metric_name": "organoid_mean_intensity_z", "metric_title": "Whole-cyst mean intensity", "reporter": "RFP"},
    {"metric_name": "organoid_mean_intensity_z", "metric_title": "Whole-cyst mean intensity", "reporter": "YFP"},
    {"metric_name": "brightest_decile_mean_intensity_z", "metric_title": "Brightest 10% mean intensity", "reporter": "RFP"},
    {"metric_name": "brightest_decile_mean_intensity_z", "metric_title": "Brightest 10% mean intensity", "reporter": "YFP"},
]

def build_population_normalized_trace_table(metric_name: str, reporter: str) -> pd.DataFrame:
    rows = []
    reporter_df = population_metrics.loc[population_metrics["reporter"] == reporter].copy()
    for position_label, position_df in reporter_df.groupby("position_label", sort=True):
        position_df = position_df.sort_values("time_hours")
        time_hours = position_df["time_hours"].to_numpy(dtype=float)
        raw_values = position_df[metric_name].to_numpy(dtype=float)
        normalized_values = globally_normalize_trace_for_display(
            raw_values,
            metric_name=metric_name,
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        finite = np.isfinite(time_hours) & np.isfinite(normalized_values)
        for time_h, value in zip(time_hours[finite], normalized_values[finite]):
            rows.append(
                {
                    "position_label": position_label,
                    "time_hours": float(time_h),
                    "normalized_value": float(value),
                }
            )
    return pd.DataFrame(rows)

def compute_mean_of_derivatives(normalized_trace_df: pd.DataFrame, sigma: float) -> pd.DataFrame:
    derivative_rows = []
    for position_label, position_df in normalized_trace_df.groupby("position_label", sort=True):
        position_df = position_df.sort_values("time_hours")
        time_hours = position_df["time_hours"].to_numpy(dtype=float)
        normalized_values = position_df["normalized_value"].to_numpy(dtype=float)
        source_values = derivative_gaussian_smooth(normalized_values, sigma)
        finite = np.isfinite(time_hours) & np.isfinite(source_values)
        if finite.sum() < 2:
            continue
        derivative_values = np.full_like(source_values, np.nan, dtype=float)
        derivative_values[finite] = local_gradient(source_values[finite], time_hours[finite])
        finite_derivative = np.isfinite(derivative_values)
        for time_h, value in zip(time_hours[finite_derivative], derivative_values[finite_derivative]):
            derivative_rows.append(
                {
                    "time_hours": float(time_h),
                    "derivative_value": float(value),
                }
            )
    if not derivative_rows:
        return pd.DataFrame(columns=["time_hours", "mean"])
    return (
        pd.DataFrame(derivative_rows)
        .groupby("time_hours", as_index=False)
        .agg(mean=("derivative_value", "mean"))
        .sort_values("time_hours")
        .reset_index(drop=True)
    )

def compute_derivative_of_mean(normalized_trace_df: pd.DataFrame, sigma: float) -> pd.DataFrame:
    if normalized_trace_df.empty:
        return pd.DataFrame(columns=["time_hours", "mean"])
    mean_trace_df = (
        normalized_trace_df.groupby("time_hours", as_index=False)
        .agg(mean=("normalized_value", "mean"))
        .sort_values("time_hours")
        .reset_index(drop=True)
    )
    mean_time = mean_trace_df["time_hours"].to_numpy(dtype=float)
    mean_values = mean_trace_df["mean"].to_numpy(dtype=float)
    mean_source_values = derivative_gaussian_smooth(mean_values, sigma)
    finite_mean = np.isfinite(mean_time) & np.isfinite(mean_source_values)
    if finite_mean.sum() < 2:
        return pd.DataFrame(columns=["time_hours", "mean"])
    derivative_values = local_gradient(mean_source_values[finite_mean], mean_time[finite_mean])
    return pd.DataFrame(
        {
            "time_hours": mean_time[finite_mean],
            "mean": derivative_values,
        }
    )

def robust_focus_ylim_from_arrays(
    arrays: list[np.ndarray],
    include_zero: bool = True,
    padding_fraction: float = 0.10,
    quantiles: tuple[float, float] = (0.02, 0.98),
) -> tuple[float, float]:
    finite_arrays = [
        np.asarray(arr, dtype=float)[np.isfinite(arr)]
        for arr in arrays
        if np.asarray(arr, dtype=float).size and np.isfinite(arr).any()
    ]
    if not finite_arrays:
        return (-1.0, 1.0)
    pooled = np.concatenate(finite_arrays)
    lower = float(np.nanquantile(pooled, quantiles[0]))
    upper = float(np.nanquantile(pooled, quantiles[1]))
    if include_zero:
        lower = min(lower, 0.0)
        upper = max(upper, 0.0)
    span = upper - lower
    if not np.isfinite(span) or span <= 0:
        span = max(abs(upper - lower), 1.0)
    pad = max(1e-6, padding_fraction * span)
    return (lower - pad, upper + pad)

def render_derivative_debug_figure(
    figure_path: Path,
    figure_title: str,
    definition_label: str,
    compute_summary_fn,
    linestyle: str,
    linewidth: float,
) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.5), sharex=True, constrained_layout=True)
    for ax, spec in zip(axes.flat, derivative_debug_specs):
        normalized_trace_df = build_population_normalized_trace_table(spec["metric_name"], spec["reporter"])
        panel_plotted = []
        if normalized_trace_df.empty:
            ax.set_visible(False)
            continue
        for sigma in derivative_debug_sigmas:
            summary_df = compute_summary_fn(normalized_trace_df, sigma)
            if summary_df.empty:
                continue
            time_hours = summary_df["time_hours"].to_numpy(dtype=float)
            values = summary_df["mean"].to_numpy(dtype=float)
            panel_plotted.append(values)
            ax.plot(
                time_hours,
                values,
                color=derivative_debug_colors[sigma],
                linewidth=(0.9 if sigma == 0 else linewidth),
                linestyle=("-" if sigma == 0 else linestyle),
                alpha=(0.32 if sigma == 0 else 1.0),
                label=("sigma = 0 (none)" if sigma == 0 else f"sigma = {int(sigma)} frames"),
            )
        ax.axhline(0.0, color="0.45", linewidth=1.45, linestyle="--", alpha=0.9)
        ax.set_title(
            next_plot_title(f"{spec['metric_title']} | {reporter_display(spec['reporter'])} | {definition_label}"),
            fontsize=10.0,
        )
        ax.set_xlabel("Time (hours)")
        set_display_time_axis(ax, "x")
        ax.set_ylabel("Mean d/dt (population-normalized progression per hour)")
        ax.grid(alpha=0.18)
        ax.set_ylim(*robust_focus_ylim_from_arrays(panel_plotted, include_zero=True, padding_fraction=0.10))
        ax.legend(loc="upper left", frameon=False, fontsize=8)
    fig.suptitle(figure_title, fontsize=11.2)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)

render_derivative_debug_figure(
    figure_path("05c_threshold_free_intensity_derivative_debug_mean_of_derivatives.png"),
    "Derivative debugging: mean of per-position derivatives",
    "mean of per-position derivatives",
    compute_mean_of_derivatives,
    "-",
    2.0,
)

render_derivative_debug_figure(
    figure_path("05c_threshold_free_intensity_derivative_debug_derivative_of_mean.png"),
    "Derivative debugging: derivative of aggregate mean trace",
    "derivative of aggregate mean trace",
    compute_derivative_of_mean,
    "--",
    1.9,
)

fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.5), sharex=True, constrained_layout=True)
for ax, spec in zip(axes.flat, derivative_debug_specs):
    normalized_trace_df = build_population_normalized_trace_table(spec["metric_name"], spec["reporter"])
    panel_plotted = []
    if normalized_trace_df.empty:
        ax.set_visible(False)
        continue
    sigma = 16.0
    mod_df = compute_mean_of_derivatives(normalized_trace_df, sigma)
    dom_df = compute_derivative_of_mean(normalized_trace_df, sigma)
    if not mod_df.empty:
        mod_time = mod_df["time_hours"].to_numpy(dtype=float)
        mod_values = mod_df["mean"].to_numpy(dtype=float)
        panel_plotted.append(mod_values)
        ax.plot(
            mod_time,
            mod_values,
            color=derivative_debug_colors[sigma],
            linewidth=2.0,
            linestyle="-",
            label="Mean of per-position derivatives",
        )
    if not dom_df.empty:
        dom_time = dom_df["time_hours"].to_numpy(dtype=float)
        dom_values = dom_df["mean"].to_numpy(dtype=float)
        panel_plotted.append(dom_values)
        ax.plot(
            dom_time,
            dom_values,
            color=derivative_debug_colors[sigma],
            linewidth=1.9,
            linestyle="--",
            label="Derivative of aggregate mean trace",
        )

    ax.axhline(0.0, color="0.45", linewidth=1.45, linestyle="--", alpha=0.9)
    ax.set_title(
        next_plot_title(f"{spec['metric_title']} | {reporter_display(spec['reporter'])} | sigma = 16 frames comparison"),
        fontsize=10.0,
    )
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel("Mean d/dt (population-normalized progression per hour)")
    ax.grid(alpha=0.18)
    ax.set_ylim(*robust_focus_ylim_from_arrays(panel_plotted, include_zero=True, padding_fraction=0.10))
    ax.legend(loc="upper left", frameon=False, fontsize=8)

derivative_debug_fig_path = figure_path("05c_threshold_free_intensity_derivative_debug_sigma16_comparison.png")
fig.suptitle("Derivative debugging: sigma = 16 frames comparison", fontsize=11.6)
fig.savefig(derivative_debug_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", derivative_debug_fig_path)


##### Experimental Area-Versus-Intensity Derivative Separation

This is a simpler exploratory comparison based on the same `05c` derivative logic as plots `11` and `12`.

For each position, using the default threshold pair (`RFP 4 sigma`, `YFP 3 sigma`), we compute:

- `A`: whole-cyst mean-intensity derivative, after putting the trace on a shared population-wide `0–1` scale
- `B`: reporter-positive area derivative, after putting the positive-fraction trace on its own shared population-wide `0–1` scale
- `C`: the exploratory residual `A - B`

The first figure shows the smoothed normalized source traces that feed into `A` and `B`.

The second figure shows the resulting derivatives in `% of total range per hour`.

All smoothing here uses the same `05c`-specific boundary-aware Gaussian smoother with no edge padding and the same `sigma = 16` frames as the main derivative section.


In [ ]:
experimental_component_specs = [
    {
        "reporter": "RFP",
        "area_metric_name": "positive_fraction_sigma4",
    },
    {
        "reporter": "YFP",
        "area_metric_name": "positive_fraction_sigma3",
    },
]

source_colors = {
    "intensity_source": "#111111",
    "area_source": "#4c78a8",
}
derivative_colors = {
    "intensity_dt": "#111111",
    "area_dt": "#4c78a8",
    "residual_dt": "#59a14f",
}

fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.9), sharex=True, constrained_layout=True)
for ax, spec in zip(axes, experimental_component_specs):
    reporter = spec["reporter"]
    source_rows = []
    for _, group_idx in population_metrics.groupby(["reporter", "position_label"], sort=True).groups.items():
        ordered = (
            population_metrics.loc[list(group_idx), ["reporter", "time_hours", "organoid_mean_intensity_z", spec["area_metric_name"]]]
            .sort_values("time_hours")
        )
        if ordered["reporter"].iloc[0] != reporter:
            continue

        time_hours = ordered["time_hours"].to_numpy(dtype=float)
        intensity_values = ordered["organoid_mean_intensity_z"].to_numpy(dtype=float)
        area_values = ordered[spec["area_metric_name"]].to_numpy(dtype=float)
        intensity_normalized = globally_normalize_trace_for_display(
            intensity_values,
            metric_name="organoid_mean_intensity_z",
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        area_normalized = globally_normalize_trace_for_display(
            area_values,
            metric_name=spec["area_metric_name"],
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        finite = np.isfinite(time_hours) & np.isfinite(intensity_normalized) & np.isfinite(area_normalized)
        if finite.sum() < 2:
            continue

        valid_time = time_hours[finite]
        intensity_source = derivative_gaussian_smooth(intensity_normalized[finite], DERIVATIVE_SMOOTH_SIGMA) * 100.0
        area_source = derivative_gaussian_smooth(area_normalized[finite], DERIVATIVE_SMOOTH_SIGMA) * 100.0

        for time_h, intensity_value, area_value in zip(
            valid_time,
            intensity_source,
            area_source,
        ):
            source_rows.append(
                {
                    "time_hours": float(time_h),
                    "intensity_source": float(intensity_value),
                    "area_source": float(area_value),
                }
            )

    source_df = pd.DataFrame(source_rows)
    if source_df.empty:
        ax.set_visible(False)
        continue

    summary = (
        source_df.groupby("time_hours", as_index=False)
        .agg(
            intensity_source=("intensity_source", "mean"),
            area_source=("area_source", "mean"),
        )
        .sort_values("time_hours")
        .reset_index(drop=True)
    )

    time_values = summary["time_hours"].to_numpy(dtype=float)
    plotted = []
    for column, label in [
        ("intensity_source", "Input A: normalized whole-cyst intensity (smoothed)"),
        ("area_source", "Input B source: normalized reporter-positive area (smoothed)"),
    ]:
        values = summary[column].to_numpy(dtype=float)
        plotted.append(values)
        ax.plot(
            time_values,
            values,
            color=source_colors[column],
            linewidth=2.4,
            label=label,
        )
    ax.set_title(next_plot_title(f"Experimental source traces for A and B | {reporter_display(reporter)}"), fontsize=9.3)
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel("% of population-wide dynamic range")
    ax.grid(alpha=0.18)
    ax.set_ylim(focus_ylim_from_arrays(plotted, include_zero=True, padding_fraction=0.10))
    ax.legend(loc="upper left", frameon=False, fontsize=8.2)

fig.suptitle("Experimental source traces for area-versus-intensity derivative separation", fontsize=11.6)
experimental_inputs_fig_path = figure_path("05c_experimental_area_adjusted_inputs.png")
fig.savefig(experimental_inputs_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", experimental_inputs_fig_path)

fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.9), sharex=True, constrained_layout=True)
for ax, spec in zip(axes, experimental_component_specs):
    reporter = spec["reporter"]
    derivative_rows = []
    for _, group_idx in population_metrics.groupby(["reporter", "position_label"], sort=True).groups.items():
        ordered = (
            population_metrics.loc[list(group_idx), ["reporter", "time_hours", "organoid_mean_intensity_z", spec["area_metric_name"]]]
            .sort_values("time_hours")
        )
        if ordered["reporter"].iloc[0] != reporter:
            continue

        time_hours = ordered["time_hours"].to_numpy(dtype=float)
        intensity_values = ordered["organoid_mean_intensity_z"].to_numpy(dtype=float)
        area_values = ordered[spec["area_metric_name"]].to_numpy(dtype=float)
        intensity_normalized = globally_normalize_trace_for_display(
            intensity_values,
            metric_name="organoid_mean_intensity_z",
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        area_normalized = globally_normalize_trace_for_display(
            area_values,
            metric_name=spec["area_metric_name"],
            reporter=reporter,
            early_n=8,
            smooth_sigma=4.0,
            clip_normalized=False,
        )
        finite = np.isfinite(time_hours) & np.isfinite(intensity_normalized) & np.isfinite(area_normalized)
        if finite.sum() < 2:
            continue

        valid_time = time_hours[finite]
        intensity_source = derivative_gaussian_smooth(intensity_normalized[finite], DERIVATIVE_SMOOTH_SIGMA)
        area_source = derivative_gaussian_smooth(area_normalized[finite], DERIVATIVE_SMOOTH_SIGMA)

        intensity_dt = local_gradient(intensity_source, valid_time) * 100.0
        area_dt = local_gradient(area_source, valid_time) * 100.0
        residual_dt = intensity_dt - area_dt

        for time_h, intensity_component, area_component, residual_component in zip(
            valid_time,
            intensity_dt,
            area_dt,
            residual_dt,
        ):
            derivative_rows.append(
                {
                    "time_hours": float(time_h),
                    "intensity_dt": float(intensity_component),
                    "area_dt": float(area_component),
                    "residual_dt": float(residual_component),
                }
            )

    derivative_df = pd.DataFrame(derivative_rows)
    if derivative_df.empty:
        ax.set_visible(False)
        continue

    summary = (
        derivative_df.groupby("time_hours", as_index=False)
        .agg(
            intensity_dt=("intensity_dt", "mean"),
            area_dt=("area_dt", "mean"),
            residual_dt=("residual_dt", "mean"),
        )
        .sort_values("time_hours")
        .reset_index(drop=True)
    )

    plotted = []
    for key, label in [
        ("intensity_dt", "A: d/dt of normalized whole-cyst intensity"),
        ("area_dt", "B: d/dt of normalized reporter-positive area"),
        ("residual_dt", "C: A - B residual"),
    ]:
        values = summary[key].to_numpy(dtype=float)
        plotted.append(values)
        ax.plot(
            summary["time_hours"].to_numpy(dtype=float),
            values,
            color=derivative_colors[key],
            linewidth=2.3,
            label=label,
        )

    ax.axhline(0.0, color="0.45", linewidth=1.45, linestyle="--", alpha=0.9)
    ax.set_title(
        next_plot_title(f"Experimental A/B/C derivative separation | {reporter_display(reporter)}"),
        fontsize=9.1,
    )
    ax.set_xlabel("Time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel("% of population-wide dynamic range per hour")
    ax.grid(alpha=0.18)
    ax.set_ylim(focus_ylim_from_arrays(plotted, include_zero=True, padding_fraction=0.10))
    ax.legend(loc="upper left", frameon=False, fontsize=8.2)

fig.suptitle("Experimental separation of intensity and area derivative terms", fontsize=11.6)
experimental_component_fig_path = figure_path("05c_experimental_area_adjusted_whole_cyst_derivative.png")
fig.savefig(experimental_component_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", experimental_component_fig_path)


##### Integrated-Intensity Views

These panels isolate the positive-region integrated-intensity summaries from the threshold-free intensity summaries above.


In [ ]:
integrated_intensity_specs = [
    {
        "metric_name": "positive_area_theme2_default",
        "title": "Absolute reporter-positive area",
        "ylabel": "Mean reporter-positive area (pixels)",
        "spread_style": "iqr",
        "include_zero": True,
        "ylim_padding_fraction": 0.16,
    },
    {
        "metric_name": "positive_integrated_intensity_theme2_default",
        "title": "Sum of corrected intensities in reporter-positive region",
        "ylabel": "Mean positive-region integrated intensity",
        "spread_style": "iqr",
        "include_zero": True,
        "ylim_padding_fraction": 0.16,
    },
    {
        "metric_name": integrated_intensity_scale_col,
        "title": "Intensity-scaled integrated intensity",
        "ylabel": "Mean intensity-scaled integrated intensity",
        "spread_style": "iqr",
        "normalization_mode_override": None,
        "include_zero": True,
        "ylim_padding_fraction": 0.16,
    },
    {
        "metric_name": integrated_intensity_meantrace_normalized_col,
        "title": "Integrated intensity normalized to aggregate mean-trace baseline and peak",
        "ylabel": "Mean normalized integrated intensity",
        "spread_style": "iqr",
        "normalization_mode_override": None,
        "include_zero": True,
        "ylim_padding_fraction": 0.16,
    },
]

render_mean_grid(
    integrated_intensity_specs,
    figure_path("05c_integrated_intensity_summaries.png"),
    "Integrated views: absolute area, raw intensity, intensity-scaled, and aggregate-normalized",
    normalization_mode=None,
    ci_level=0.95,
)


#### Cyst-Preserving Joint Reporter Trajectories

The SD bands above show marginal spread for `RFP` and `YFP` separately.

The phase-plane view below keeps cyst identity intact:

- each thin colored trajectory is one position
- color encodes absolute time
- the bold black path is the population-mean trajectory through the same space


In [ ]:
phase_metric_name_by_reporter = {
    "RFP": "positive_fraction_sigma4",
    "YFP": "positive_fraction_sigma3",
}

time_min = float(population_metrics["time_hours"].min())
time_max = float(population_metrics["time_hours"].max())
time_norm = Normalize(vmin=time_min, vmax=time_max)
phase_cmap = plt.cm.viridis
phase_joint_df = build_joint_metric_table(phase_metric_name_by_reporter)

fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)

for position_label, joint in phase_joint_df.groupby("position_label", sort=True):
    if len(joint) < 3:
        continue

    x = joint["rfp_value"].to_numpy(dtype=float)
    y = joint["yfp_value"].to_numpy(dtype=float)
    t = joint["time_hours"].to_numpy(dtype=float)
    points = np.column_stack([x, y]).reshape(-1, 1, 2)
    segments = np.concatenate([points[:-1], points[1:]], axis=1)
    segment_times = 0.5 * (t[:-1] + t[1:])

    lc = LineCollection(
        segments,
        cmap=phase_cmap,
        norm=time_norm,
        linewidths=0.9,
        alpha=0.10,
    )
    lc.set_array(segment_times)
    ax.add_collection(lc)

mean_rfp = aggregate_display_trace(phase_metric_name_by_reporter["RFP"], "RFP", normalization_mode=None)
mean_yfp = aggregate_display_trace(phase_metric_name_by_reporter["YFP"], "YFP", normalization_mode=None)
mean_joint = (
    mean_rfp[["time_hours", "mean"]]
    .rename(columns={"mean": "rfp_mean"})
    .merge(
        mean_yfp[["time_hours", "mean"]].rename(columns={"mean": "yfp_mean"}),
        on="time_hours",
        how="inner",
    )
    .sort_values("time_hours")
)

ax.plot(
    mean_joint["rfp_mean"],
    mean_joint["yfp_mean"],
    color="white",
    linewidth=6.8,
    alpha=0.95,
    zorder=4,
)

ax.plot(
    mean_joint["rfp_mean"],
    mean_joint["yfp_mean"],
    color="#111111",
    linewidth=4.7,
    label="population mean trajectory",
    zorder=5,
)

anchor_times = np.array([0, 8, 16, 24, 32, 40, 48, 56, 64], dtype=float)
anchor_rows = []
for target_time in anchor_times:
    idx = np.argmin(np.abs(mean_joint["time_hours"].to_numpy(dtype=float) - target_time))
    anchor_rows.append(mean_joint.iloc[int(idx)])
anchor_rows.append(mean_joint.iloc[-1])
anchor_df = pd.DataFrame(anchor_rows).drop_duplicates(subset=["time_hours"])
ax.scatter(
    anchor_df["rfp_mean"],
    anchor_df["yfp_mean"],
    c=anchor_df["time_hours"],
    cmap=phase_cmap,
    norm=time_norm,
    s=34,
    edgecolor="white",
    linewidth=0.5,
    zorder=6,
)

ax.set_xlabel(display_text("RFP positive fraction"))
ax.set_ylabel(display_text("YFP positive fraction"))
ax.set_title(next_plot_title("Positive-fraction phase plane with cyst identity preserved"), fontsize=10.2)
ax.grid(alpha=0.18)
ax.legend(loc="upper left", frameon=False)

x_arrays = []
y_arrays = []
for position_label, joint in phase_joint_df.groupby("position_label", sort=True):
    if len(joint) >= 3:
        x_arrays.append(joint["rfp_value"].to_numpy(dtype=float))
        y_arrays.append(joint["yfp_value"].to_numpy(dtype=float))
x_arrays.append(mean_joint["rfp_mean"].to_numpy(dtype=float))
y_arrays.append(mean_joint["yfp_mean"].to_numpy(dtype=float))
ax.set_xlim(focus_ylim_from_arrays(x_arrays, include_zero=True, padding_fraction=0.08))
ax.set_ylim(focus_ylim_from_arrays(y_arrays, include_zero=True, padding_fraction=0.08))

sm = plt.cm.ScalarMappable(norm=time_norm, cmap=phase_cmap)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, label="Time (hours)")
set_display_time_colorbar(cbar)

phase_plane_path = figure_path("05c_positive_fraction_phase_plane.png")
fig.suptitle(display_text("Cyst-preserving joint RFP/YFP trajectory view"), fontsize=11.6)
fig.savefig(phase_plane_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", phase_plane_path)


#### Cleaner Joint Alternatives

The full trajectory view above preserves identity, but it is also cluttered.

These alternatives keep actual trajectories while simplifying the display:

- 4-hour binned trajectories across all cysts
- a threshold-free intensity analogue
- a representative subset of cysts shown separately


In [ ]:
def render_binned_phase_plane(
    joint_df: pd.DataFrame,
    mean_x: pd.DataFrame,
    mean_y: pd.DataFrame,
    figure_path: Path,
    figure_title: str,
    x_label: str,
    y_label: str,
    bin_width_hours: float = 4.0,
    display_transform: str | None = None,
    shifted_log_axis_quantiles: tuple[float, float] = (0.01, 0.99),
    shifted_log_shift_quantile: float = 0.01,
) -> None:
    binned_df = bin_joint_trajectory(joint_df, bin_width_hours=bin_width_hours)
    fig, ax = plt.subplots(figsize=(6.2, 5.4), constrained_layout=True)

    mean_joint = build_mean_joint_path(mean_x, mean_y)
    mean_joint_binned = (
        mean_joint.assign(
            time_bin_center=(
                np.floor(mean_joint["time_hours"].to_numpy(dtype=float) / float(bin_width_hours)) * float(bin_width_hours)
                + float(bin_width_hours) / 2.0
            )
        )
        .groupby("time_bin_center", as_index=False)
        .agg(
            rfp_mean=("rfp_mean", "mean"),
            yfp_mean=("yfp_mean", "mean"),
        )
        .sort_values("time_bin_center")
    )

    if display_transform == "shifted_log":
        x_values_for_scale = np.concatenate(
            [
                binned_df["rfp_value"].to_numpy(dtype=float),
                mean_joint_binned["rfp_mean"].to_numpy(dtype=float),
            ]
        )
        y_values_for_scale = np.concatenate(
            [
                binned_df["yfp_value"].to_numpy(dtype=float),
                mean_joint_binned["yfp_mean"].to_numpy(dtype=float),
            ]
        )
        x_shift = float(np.nanquantile(x_values_for_scale[np.isfinite(x_values_for_scale)], shifted_log_shift_quantile))
        y_shift = float(np.nanquantile(y_values_for_scale[np.isfinite(y_values_for_scale)], shifted_log_shift_quantile))
        x_eps = max(1e-3, 0.02 * max(float(np.nanpercentile(x_values_for_scale, 95) - x_shift), 1.0))
        y_eps = max(1e-3, 0.02 * max(float(np.nanpercentile(y_values_for_scale, 95) - y_shift), 1.0))

        def transform_x(values: np.ndarray) -> np.ndarray:
            arr = np.asarray(values, dtype=float)
            return np.log1p(np.maximum(arr - x_shift + x_eps, 0.0))

        def transform_y(values: np.ndarray) -> np.ndarray:
            arr = np.asarray(values, dtype=float)
            return np.log1p(np.maximum(arr - y_shift + y_eps, 0.0))
        mean_rfp_values = transform_x(mean_joint_binned["rfp_mean"].to_numpy(dtype=float))
        mean_yfp_values = transform_y(mean_joint_binned["yfp_mean"].to_numpy(dtype=float))
        x_arrays = [transform_x(binned_df["rfp_value"].to_numpy(dtype=float)), mean_rfp_values]
        y_arrays = [transform_y(binned_df["yfp_value"].to_numpy(dtype=float)), mean_yfp_values]
        x_label = f"Shifted-log {x_label.lower()}"
        y_label = f"Shifted-log {y_label.lower()}"
    elif display_transform == "signed_log":
        x_values_for_scale = np.concatenate(
            [
                binned_df["rfp_value"].to_numpy(dtype=float),
                mean_joint_binned["rfp_mean"].to_numpy(dtype=float),
            ]
        )
        y_values_for_scale = np.concatenate(
            [
                binned_df["yfp_value"].to_numpy(dtype=float),
                mean_joint_binned["yfp_mean"].to_numpy(dtype=float),
            ]
        )
        x_abs_scale = max(
            0.15,
            float(np.nanquantile(np.abs(x_values_for_scale[np.isfinite(x_values_for_scale)]), 0.25)),
        )
        y_abs_scale = max(
            0.15,
            float(np.nanquantile(np.abs(y_values_for_scale[np.isfinite(y_values_for_scale)]), 0.25)),
        )

        def transform_x(values: np.ndarray) -> np.ndarray:
            arr = np.asarray(values, dtype=float)
            return np.sign(arr) * np.log1p(np.abs(arr) / x_abs_scale)

        def transform_y(values: np.ndarray) -> np.ndarray:
            arr = np.asarray(values, dtype=float)
            return np.sign(arr) * np.log1p(np.abs(arr) / y_abs_scale)

        mean_rfp_values = transform_x(mean_joint_binned["rfp_mean"].to_numpy(dtype=float))
        mean_yfp_values = transform_y(mean_joint_binned["yfp_mean"].to_numpy(dtype=float))
        x_arrays = [transform_x(binned_df["rfp_value"].to_numpy(dtype=float)), mean_rfp_values]
        y_arrays = [transform_y(binned_df["yfp_value"].to_numpy(dtype=float)), mean_yfp_values]
        x_label = f"Signed-log {x_label.lower()}"
        y_label = f"Signed-log {y_label.lower()}"
    else:
        def transform_x(values: np.ndarray) -> np.ndarray:
            return np.asarray(values, dtype=float)

        def transform_y(values: np.ndarray) -> np.ndarray:
            return np.asarray(values, dtype=float)

        mean_rfp_values = mean_joint_binned["rfp_mean"].to_numpy(dtype=float)
        mean_yfp_values = mean_joint_binned["yfp_mean"].to_numpy(dtype=float)
        x_arrays = [binned_df["rfp_value"].to_numpy(dtype=float), mean_rfp_values]
        y_arrays = [binned_df["yfp_value"].to_numpy(dtype=float), mean_yfp_values]

    for position_label, joint in binned_df.groupby("position_label", sort=True):
        if len(joint) < 2:
            continue
        x = transform_x(joint["rfp_value"].to_numpy(dtype=float))
        y = transform_y(joint["yfp_value"].to_numpy(dtype=float))
        t = joint["time_bin_center"].to_numpy(dtype=float)
        points = np.column_stack([x, y]).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)
        segment_times = 0.5 * (t[:-1] + t[1:])
        lc = LineCollection(
            segments,
            cmap=phase_cmap,
            norm=time_norm,
            linewidths=0.8,
            alpha=0.12,
        )
        lc.set_array(segment_times)
        ax.add_collection(lc)

    ax.plot(
        mean_rfp_values,
        mean_yfp_values,
        color="white",
        linewidth=5.6,
        alpha=0.96,
        zorder=4,
    )
    ax.plot(
        mean_rfp_values,
        mean_yfp_values,
        color="#111111",
        linewidth=3.9,
        label=f"population mean ({int(bin_width_hours)} h bins)",
        zorder=5,
    )
    ax.scatter(
        mean_rfp_values,
        mean_yfp_values,
        c=mean_joint_binned["time_bin_center"],
        cmap=phase_cmap,
        norm=time_norm,
        s=36,
        edgecolor="white",
        linewidth=0.5,
        zorder=6,
    )

    axis_include_zero = False if display_transform == "shifted_log" else True
    if display_transform == "shifted_log":
        ax.set_xlim(
            robust_focus_ylim_from_arrays(
                x_arrays,
                include_zero=False,
                padding_fraction=0.06,
                quantiles=shifted_log_axis_quantiles,
            )
        )
        ax.set_ylim(
            robust_focus_ylim_from_arrays(
                y_arrays,
                include_zero=False,
                padding_fraction=0.06,
                quantiles=shifted_log_axis_quantiles,
            )
        )
    elif display_transform == "signed_log":
        ax.set_xlim(focus_ylim_from_arrays(x_arrays, include_zero=True, padding_fraction=0.06))
        ax.set_ylim(focus_ylim_from_arrays(y_arrays, include_zero=True, padding_fraction=0.06))
    else:
        ax.set_xlim(focus_ylim_from_arrays(x_arrays, include_zero=axis_include_zero, padding_fraction=0.08))
        ax.set_ylim(focus_ylim_from_arrays(y_arrays, include_zero=axis_include_zero, padding_fraction=0.08))
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(next_plot_title(f"{figure_title} ({int(bin_width_hours)} h bins)"), fontsize=10.2)
    ax.grid(alpha=0.18)
    ax.legend(loc="upper left", frameon=False)

    sm = plt.cm.ScalarMappable(norm=time_norm, cmap=phase_cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, label="Time (hours)")
    set_display_time_colorbar(cbar)
    fig.suptitle(figure_title, fontsize=11.2)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def render_velocity_field(
    joint_df: pd.DataFrame,
    mean_x: pd.DataFrame,
    mean_y: pd.DataFrame,
    figure_path: Path,
    figure_title: str,
    x_label: str,
    y_label: str,
    bin_width_hours: float = 4.0,
    time_windows: list[tuple[float, float]] | None = None,
    grid_bins: int = 8,
    min_segments_per_bin: int = 3,
) -> None:
    segment_df = build_velocity_segments(joint_df, bin_width_hours=bin_width_hours)
    if segment_df.empty:
        display(Markdown("No valid trajectory segments were available for the velocity-field view."))
        return

    mean_joint = build_mean_joint_path(mean_x, mean_y)
    mean_joint_binned = (
        mean_joint.assign(
            time_bin_center=(
                np.floor(mean_joint["time_hours"].to_numpy(dtype=float) / float(bin_width_hours)) * float(bin_width_hours)
                + float(bin_width_hours) / 2.0
            )
        )
        .groupby("time_bin_center", as_index=False)
        .agg(
            rfp_mean=("rfp_mean", "mean"),
            yfp_mean=("yfp_mean", "mean"),
        )
        .sort_values("time_bin_center")
    )

    if time_windows is None:
        time_windows = [(0.0, 12.0), (12.0, 24.0), (24.0, 36.0), (36.0, 48.0), (48.0, 72.0)]

    x_limits = focus_ylim_from_arrays(
        [joint_df["rfp_value"].to_numpy(dtype=float), mean_joint_binned["rfp_mean"].to_numpy(dtype=float)],
        include_zero=True,
        padding_fraction=0.08,
    )
    y_limits = focus_ylim_from_arrays(
        [joint_df["yfp_value"].to_numpy(dtype=float), mean_joint_binned["yfp_mean"].to_numpy(dtype=float)],
        include_zero=True,
        padding_fraction=0.08,
    )
    x_edges = np.linspace(x_limits[0], x_limits[1], grid_bins + 1)
    y_edges = np.linspace(y_limits[0], y_limits[1], grid_bins + 1)

    n_panels = len(time_windows)
    n_cols = 3
    n_rows = int(np.ceil(n_panels / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.8 * n_cols, 4.1 * n_rows), constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()
    arrow_color = "#2f6c8f"
    display_scale_hours = float(bin_width_hours)

    for ax_idx, (window_start, window_end) in enumerate(time_windows):
        ax = axes[ax_idx]
        window_df = segment_df.loc[
            (segment_df["time_mid"] >= float(window_start))
            & (segment_df["time_mid"] < float(window_end))
        ].copy()

        ax.plot(
            mean_joint_binned["rfp_mean"],
            mean_joint_binned["yfp_mean"],
            color="0.72",
            linewidth=1.5,
            zorder=1,
        )
        mean_window = mean_joint_binned.loc[
            (mean_joint_binned["time_bin_center"] >= float(window_start))
            & (mean_joint_binned["time_bin_center"] < float(window_end))
        ]
        if len(mean_window) >= 1:
            ax.plot(
                mean_window["rfp_mean"],
                mean_window["yfp_mean"],
                color="#111111",
                linewidth=2.3,
                zorder=3,
            )
            ax.scatter(
                mean_window["rfp_mean"],
                mean_window["yfp_mean"],
                color="#111111",
                s=18,
                zorder=4,
            )

        if not window_df.empty:
            x_bin = np.digitize(window_df["x_mid"].to_numpy(dtype=float), x_edges) - 1
            y_bin = np.digitize(window_df["y_mid"].to_numpy(dtype=float), y_edges) - 1
            valid = (
                (x_bin >= 0)
                & (x_bin < grid_bins)
                & (y_bin >= 0)
                & (y_bin < grid_bins)
            )
            grouped = (
                window_df.loc[valid]
                .assign(x_bin=x_bin[valid], y_bin=y_bin[valid])
                .groupby(["x_bin", "y_bin"], as_index=False)
                .agg(
                    x_mid=("x_mid", "mean"),
                    y_mid=("y_mid", "mean"),
                    u_per_hour=("u_per_hour", "mean"),
                    v_per_hour=("v_per_hour", "mean"),
                    n_segments=("position_label", "size"),
                )
            )
            grouped = grouped.loc[grouped["n_segments"] >= int(min_segments_per_bin)].copy()
            if not grouped.empty:
                ax.quiver(
                    grouped["x_mid"],
                    grouped["y_mid"],
                    grouped["u_per_hour"] * display_scale_hours,
                    grouped["v_per_hour"] * display_scale_hours,
                    angles="xy",
                    scale_units="xy",
                    scale=1.0,
                    color=arrow_color,
                    width=0.0045,
                    headwidth=4.0,
                    headlength=5.0,
                    alpha=0.92,
                    zorder=5,
                )
                ax.scatter(
                    grouped["x_mid"],
                    grouped["y_mid"],
                    s=10 + 4 * grouped["n_segments"].to_numpy(dtype=float),
                    color=arrow_color,
                    alpha=0.18,
                    linewidth=0.0,
                    zorder=2,
                )

        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_title(next_plot_title(f"{window_start:.0f}-{window_end:.0f} h"), fontsize=9.8)
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.grid(alpha=0.18)

    for ax in axes[n_panels:]:
        ax.axis("off")

    fig.suptitle(
        figure_title + f" (arrows show mean ~{int(display_scale_hours)} h displacement)",
        fontsize=11.6,
    )
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def choose_forward_transition_timepoints(
    joint_df: pd.DataFrame,
    n_timepoints: int = 5,
    candidate_step_hours: float = 8.0,
    min_positions: int = 40,
    return_details: bool = False,
):
    unique_times = np.sort(joint_df["time_hours"].dropna().unique().astype(float))
    if unique_times.size <= n_timepoints:
        selected_times = [float(value) for value in unique_times.tolist()]
        if return_details:
            return {
                "selected_times": selected_times,
                "transition_df": pd.DataFrame(),
                "candidate_step_hours": float(candidate_step_hours),
                "min_positions": int(min_positions),
                "score_min_aggregate_length": float("nan"),
                "score_total_aggregate_length": float("nan"),
                "score_mean_individual_displacement": float("nan"),
            }
        return selected_times

    candidate_times = [
        float(value)
        for value in unique_times
        if np.isclose(value / candidate_step_hours, np.round(value / candidate_step_hours))
    ]
    if not candidate_times or not np.isclose(candidate_times[0], unique_times[0]):
        candidate_times = [float(unique_times[0])] + candidate_times
    if not np.isclose(candidate_times[-1], unique_times[-1]):
        candidate_times = candidate_times + [float(unique_times[-1])]
    candidate_times = sorted(set(candidate_times))

    mean_joint = (
        joint_df.groupby("time_hours", as_index=False)
        .agg(rfp_mean=("rfp_value", "mean"), yfp_mean=("yfp_value", "mean"))
        .sort_values("time_hours")
    )
    mean_joint_lookup = {
        float(row["time_hours"]): (float(row["rfp_mean"]), float(row["yfp_mean"]))
        for _, row in mean_joint.iterrows()
    }
    max_positions = float(joint_df["position_label"].nunique())
    best_score = None
    best_times = None
    best_transition_rows = []

    for interior_times in itertools.combinations(candidate_times[1:-1], n_timepoints - 2):
        selected_times = [candidate_times[0], *interior_times, candidate_times[-1]]
        aggregate_pair_scores = []
        individual_pair_scores = []
        transition_rows = []
        valid = True
        for start_time, end_time in zip(selected_times[:-1], selected_times[1:]):
            start_df = (
                joint_df.loc[joint_df["time_hours"].eq(start_time), ["position_label", "rfp_value", "yfp_value"]]
                .rename(columns={"rfp_value": "x0", "yfp_value": "y0"})
            )
            end_df = (
                joint_df.loc[joint_df["time_hours"].eq(end_time), ["position_label", "rfp_value", "yfp_value"]]
                .rename(columns={"rfp_value": "x1", "yfp_value": "y1"})
            )
            merged = start_df.merge(end_df, on="position_label", how="inner")
            if len(merged) < min_positions:
                valid = False
                break
            displacement = np.hypot(
                merged["x1"].to_numpy(dtype=float) - merged["x0"].to_numpy(dtype=float),
                merged["y1"].to_numpy(dtype=float) - merged["y0"].to_numpy(dtype=float),
            )
            mean_individual_displacement = float(np.nanmean(displacement)) * np.sqrt(len(merged) / max_positions)
            aggregate_length = float(
                np.hypot(
                    mean_joint_lookup[end_time][0] - mean_joint_lookup[start_time][0],
                    mean_joint_lookup[end_time][1] - mean_joint_lookup[start_time][1],
                )
            )
            individual_pair_scores.append(mean_individual_displacement)
            aggregate_pair_scores.append(aggregate_length)
            transition_rows.append(
                {
                    "start_time_hours": float(start_time),
                    "end_time_hours": float(end_time),
                    "aggregate_vector_length": aggregate_length,
                    "mean_individual_displacement": mean_individual_displacement,
                    "n_positions": int(len(merged)),
                }
            )
        if not valid:
            continue
        score = (
            float(np.nanmin(aggregate_pair_scores)),
            float(np.nansum(aggregate_pair_scores)),
            float(np.nanmean(individual_pair_scores)),
        )
        if best_score is None or score > best_score:
            best_score = score
            best_times = selected_times
            best_transition_rows = transition_rows

    if best_times is None:
        fallback_idx = np.linspace(0, len(unique_times) - 1, n_timepoints, dtype=int)
        best_times = [float(unique_times[idx]) for idx in fallback_idx]
        best_transition_rows = []
    if return_details:
        transition_df = pd.DataFrame(best_transition_rows)
        if not transition_df.empty:
            transition_df.insert(0, "transition_rank", np.arange(1, len(transition_df) + 1))
        return {
            "selected_times": best_times,
            "transition_df": transition_df,
            "candidate_step_hours": float(candidate_step_hours),
            "min_positions": int(min_positions),
            "score_min_aggregate_length": float(best_score[0]) if best_score is not None else float("nan"),
            "score_total_aggregate_length": float(best_score[1]) if best_score is not None else float("nan"),
            "score_mean_individual_displacement": float(best_score[2]) if best_score is not None else float("nan"),
        }
    return best_times


def render_timepoint_vector_panels(
    joint_df: pd.DataFrame,
    figure_path: Path,
    figure_title: str,
    x_label: str,
    y_label: str,
    selected_times: list[float],
    x_limits: tuple[float, float] = (0.0, 1.0),
    y_limits: tuple[float, float] = (0.0, 1.0),
) -> None:
    transition_labels = [
        f"{display_time_hours(start_time):g} -> {display_time_hours(end_time):g} h"
        for start_time, end_time in zip(selected_times[:-1], selected_times[1:])
    ]
    fig, axes = plt.subplots(2, 2, figsize=(7.8, 6.9), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    panel_colors = [phase_cmap(time_norm(start_time)) for start_time in selected_times[:-1]]

    for idx, (ax, start_time, end_time) in enumerate(zip(axes, selected_times[:-1], selected_times[1:])):
        start_df = (
            joint_df.loc[joint_df["time_hours"].eq(start_time), ["position_label", "rfp_value", "yfp_value"]]
            .rename(columns={"rfp_value": "x0", "yfp_value": "y0"})
        )
        end_df = (
            joint_df.loc[joint_df["time_hours"].eq(end_time), ["position_label", "rfp_value", "yfp_value"]]
            .rename(columns={"rfp_value": "x1", "yfp_value": "y1"})
        )
        arrow_df = start_df.merge(end_df, on="position_label", how="inner")
        if arrow_df.empty:
            ax.axis("off")
            continue

        ax.scatter(
            arrow_df["x0"],
            arrow_df["y0"],
            s=52,
            color=panel_colors[idx],
            alpha=0.82,
            linewidth=0.0,
            zorder=1,
        )
        ax.quiver(
            arrow_df["x0"],
            arrow_df["y0"],
            arrow_df["x1"] - arrow_df["x0"],
            arrow_df["y1"] - arrow_df["y0"],
            angles="xy",
            scale_units="xy",
            scale=1.0,
            color="#111111",
            width=0.0048,
            headwidth=6.1,
            headlength=7.1,
            alpha=0.86,
            zorder=3,
        )
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_xticks([0.0, 1.0])
        ax.set_yticks([0.0, 1.0])
        ax.set_xticks(np.linspace(x_limits[0], x_limits[1], 6), minor=True)
        ax.set_yticks(np.linspace(y_limits[0], y_limits[1], 6), minor=True)
        ax.grid(alpha=0.18, which="major")
        ax.grid(alpha=0.12, which="minor")
        ax.label_outer()
        legend_handle = Line2D([0], [0], marker="o", color=panel_colors[idx], markersize=5.0, linewidth=0.0)
        ax.legend(
            [legend_handle],
            [transition_labels[idx]],
            loc="upper left",
            frameon=False,
            fontsize=8.6,
            handlelength=0.6,
            handletextpad=0.35,
            borderaxespad=0.2,
        )

    fig.supxlabel(x_label, fontsize=8.6)
    fig.supylabel(y_label, fontsize=8.6)
    sm = plt.cm.ScalarMappable(norm=time_norm, cmap=phase_cmap)
    sm.set_array([])
    cax = fig.add_axes([0.92, 0.15, 0.018, 0.68])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Time (hours)")
    set_display_time_colorbar(cbar)
    fig.suptitle(next_plot_title(figure_title), fontsize=10.8)
    fig.subplots_adjust(left=0.09, right=0.885, bottom=0.08, top=0.93, wspace=0.06, hspace=0.08)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def render_timepoint_density_aggregate_panels(
    joint_df: pd.DataFrame,
    figure_path: Path,
    figure_title: str,
    x_label: str,
    y_label: str,
    selected_times: list[float],
    x_limits: tuple[float, float] = (0.0, 1.0),
    y_limits: tuple[float, float] = (0.0, 1.0),
) -> None:
    transition_labels = [
        f"{display_time_hours(start_time):g} -> {display_time_hours(end_time):g} h"
        for start_time, end_time in zip(selected_times[:-1], selected_times[1:])
    ]
    fig, axes = plt.subplots(2, 2, figsize=(7.8, 6.9), sharex=True, sharey=True)
    axes = np.atleast_1d(axes).ravel()
    panel_colors = [phase_cmap(time_norm(start_time)) for start_time in selected_times[:-1]]

    for idx, (ax, start_time, end_time) in enumerate(zip(axes, selected_times[:-1], selected_times[1:])):
        start_df = (
            joint_df.loc[joint_df["time_hours"].eq(start_time), ["position_label", "rfp_value", "yfp_value"]]
            .rename(columns={"rfp_value": "x0", "yfp_value": "y0"})
        )
        end_df = (
            joint_df.loc[joint_df["time_hours"].eq(end_time), ["position_label", "rfp_value", "yfp_value"]]
            .rename(columns={"rfp_value": "x1", "yfp_value": "y1"})
        )
        arrow_df = start_df.merge(end_df, on="position_label", how="inner")
        if arrow_df.empty:
            ax.axis("off")
            continue
        try:
            if len(arrow_df) >= 8:
                values = np.vstack([
                    arrow_df["x0"].to_numpy(dtype=float),
                    arrow_df["y0"].to_numpy(dtype=float),
                ])
                kde = stats.gaussian_kde(values)
                xx, yy = np.meshgrid(
                    np.linspace(x_limits[0], x_limits[1], 90),
                    np.linspace(y_limits[0], y_limits[1], 90),
                )
                density = kde(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
                ax.contourf(
                    xx,
                    yy,
                    density,
                    levels=6,
                    cmap=LinearSegmentedColormap.from_list(
                        f"timepoint_{idx}",
                        ["#ffffff", panel_colors[idx]],
                    ),
                    alpha=0.74,
                    zorder=1,
                )
        except Exception:
            pass

        ax.scatter(
            arrow_df["x0"],
            arrow_df["y0"],
            s=34,
            color=panel_colors[idx],
            alpha=0.74,
            linewidth=0.0,
            zorder=2,
        )
        mean_x0 = float(np.nanmean(arrow_df["x0"]))
        mean_y0 = float(np.nanmean(arrow_df["y0"]))
        mean_dx = float(np.nanmean(arrow_df["x1"] - arrow_df["x0"]))
        mean_dy = float(np.nanmean(arrow_df["y1"] - arrow_df["y0"]))
        ax.quiver(
            mean_x0,
            mean_y0,
            mean_dx,
            mean_dy,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            color="#111111",
            width=0.0093,
            headwidth=8.0,
            headlength=9.0,
            alpha=1.0,
            zorder=4,
        )
        ax.scatter(
            [mean_x0],
            [mean_y0],
            s=110,
            color="#111111",
            edgecolor="white",
            linewidth=0.9,
            zorder=5,
        )

        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_xticks([0.0, 1.0])
        ax.set_yticks([0.0, 1.0])
        ax.set_xticks(np.linspace(x_limits[0], x_limits[1], 6), minor=True)
        ax.set_yticks(np.linspace(y_limits[0], y_limits[1], 6), minor=True)
        ax.grid(alpha=0.18, which="major")
        ax.grid(alpha=0.12, which="minor")
        ax.label_outer()
        legend_handle = Patch(facecolor=panel_colors[idx], edgecolor="none", alpha=0.65)
        ax.legend(
            [legend_handle],
            [transition_labels[idx]],
            loc="upper left",
            frameon=False,
            fontsize=8.6,
            handlelength=0.9,
            handletextpad=0.35,
            borderaxespad=0.2,
        )

    fig.supxlabel(x_label, fontsize=8.6)
    fig.supylabel(y_label, fontsize=8.6)
    sm = plt.cm.ScalarMappable(norm=time_norm, cmap=phase_cmap)
    sm.set_array([])
    cax = fig.add_axes([0.92, 0.15, 0.018, 0.68])
    cbar = fig.colorbar(sm, cax=cax)
    cbar.set_label("Time (hours)")
    set_display_time_colorbar(cbar)
    fig.suptitle(next_plot_title(figure_title), fontsize=10.8)
    fig.subplots_adjust(left=0.09, right=0.885, bottom=0.08, top=0.93, wspace=0.06, hspace=0.08)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


def render_representative_phase_grid(
    joint_df: pd.DataFrame,
    figure_path: Path,
    figure_title: str,
    x_label: str,
    y_label: str,
    position_labels: list[str],
    bin_width_hours: float = 4.0,
) -> None:
    subset_df = joint_df.loc[joint_df["position_label"].isin(position_labels)].copy()
    binned_df = bin_joint_trajectory(subset_df, bin_width_hours=bin_width_hours)
    fig, axes = plt.subplots(3, 4, figsize=(10.8, 7.1), sharex=True, sharey=True)
    axes = axes.ravel()
    x_limits = focus_ylim_from_arrays([subset_df["rfp_value"].to_numpy(dtype=float)], include_zero=True, padding_fraction=0.03)
    y_limits = focus_ylim_from_arrays([subset_df["yfp_value"].to_numpy(dtype=float)], include_zero=True, padding_fraction=0.03)
    color_handle = None

    for ax, position_label in zip(axes, position_labels):
        joint = binned_df.loc[binned_df["position_label"] == position_label].sort_values("time_bin_center")
        if joint.empty:
            ax.axis("off")
            continue
        x = joint["rfp_value"].to_numpy(dtype=float)
        y = joint["yfp_value"].to_numpy(dtype=float)
        t = joint["time_bin_center"].to_numpy(dtype=float)
        ax.plot(x, y, color="0.45", linewidth=1.2, alpha=0.8)
        scatter = ax.scatter(
            x,
            y,
            c=t,
            cmap=phase_cmap,
            norm=time_norm,
            s=26,
            edgecolor="white",
            linewidth=0.35,
        )
        color_handle = scatter
        if len(x) > 0:
            ax.scatter([x[0]], [y[0]], s=26, facecolor="none", edgecolor="#111111", linewidth=0.9, zorder=5)
            ax.scatter([x[-1]], [y[-1]], s=30, color="#111111", edgecolor="white", linewidth=0.4, zorder=5)
        rfp_range = float(np.nanmax(x) - np.nanmin(x)) if len(x) else float("nan")
        yfp_range = float(np.nanmax(y) - np.nanmin(y)) if len(y) else float("nan")
        line_handle = Line2D([0], [0], color="0.45", linewidth=1.2)
        ax.legend(
            [line_handle],
            [f"{position_label} | ΔR={rfp_range:.2f}, ΔY={yfp_range:.2f}"],
            loc="upper left",
            frameon=False,
            fontsize=6.8,
            handlelength=1.0,
            handletextpad=0.3,
            borderaxespad=0.15,
        )
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.grid(alpha=0.16)
        ax.tick_params(labelsize=6.0, length=2.0, pad=1.4)

    fig.suptitle(next_plot_title(figure_title), fontsize=11.2)
    for ax in axes:
        ax.label_outer()
    fig.supxlabel(x_label, fontsize=8.6)
    fig.supylabel(y_label, fontsize=8.6)
    if color_handle is not None:
        cax = fig.add_axes([0.92, 0.15, 0.018, 0.68])
        cbar = fig.colorbar(color_handle, cax=cax)
        cbar.set_label("Time (hours)")
        set_display_time_colorbar(cbar)
    fig.subplots_adjust(left=0.075, right=0.89, bottom=0.075, top=0.92, wspace=0.05, hspace=0.08)
    fig.savefig(figure_path, dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
    print("Wrote figure:", figure_path)


positive_fraction_mean_x = aggregate_display_trace("positive_fraction_sigma4", "RFP", normalization_mode=None)
positive_fraction_mean_y = aggregate_display_trace("positive_fraction_sigma3", "YFP", normalization_mode=None)
render_binned_phase_plane(
    joint_df=phase_joint_df,
    mean_x=positive_fraction_mean_x,
    mean_y=positive_fraction_mean_y,
    figure_path=figure_path("05c_positive_fraction_phase_plane_binned.png"),
    figure_title="Positive-fraction joint reporter trajectory view",
    x_label=display_text("RFP positive fraction"),
    y_label=display_text("YFP positive fraction"),
    bin_width_hours=4.0,
)

threshold_free_metric_name_by_reporter = {
    "RFP": "organoid_mean_intensity_z",
    "YFP": "organoid_mean_intensity_z",
}
global_normalized_threshold_free_joint_df = build_joint_metric_table(
    threshold_free_metric_name_by_reporter,
    normalization_mode="global",
    smooth_sigma=4.0,
    clip_normalized=False,
)
global_normalized_threshold_free_x = aggregate_display_trace(
    "organoid_mean_intensity_z",
    "RFP",
    normalization_mode="global",
    clip_normalized=False,
)
global_normalized_threshold_free_y = aggregate_display_trace(
    "organoid_mean_intensity_z",
    "YFP",
    normalization_mode="global",
    clip_normalized=False,
)
render_binned_phase_plane(
    joint_df=global_normalized_threshold_free_joint_df,
    mean_x=global_normalized_threshold_free_x,
    mean_y=global_normalized_threshold_free_y,
    figure_path=figure_path("05c_threshold_free_phase_plane_binned_global_normalized.png"),
    figure_title="Population-wide normalized whole-cyst intensity trajectories",
    x_label=display_text("RFP globally normalized whole-cyst intensity"),
    y_label=display_text("YFP globally normalized whole-cyst intensity"),
    bin_width_hours=4.0,
    display_transform="shifted_log",
)

per_cyst_normalized_threshold_free_joint_df = build_joint_metric_table(
    threshold_free_metric_name_by_reporter,
    normalization_mode="per_position",
    smooth_sigma=4.0,
    clip_normalized=False,
)
per_cyst_normalized_threshold_free_x = aggregate_display_trace(
    "organoid_mean_intensity_z",
    "RFP",
    normalization_mode="per_position",
    clip_normalized=False,
)
per_cyst_normalized_threshold_free_y = aggregate_display_trace(
    "organoid_mean_intensity_z",
    "YFP",
    normalization_mode="per_position",
    clip_normalized=False,
)
render_binned_phase_plane(
    joint_df=per_cyst_normalized_threshold_free_joint_df,
    mean_x=per_cyst_normalized_threshold_free_x,
    mean_y=per_cyst_normalized_threshold_free_y,
    figure_path=figure_path("05c_threshold_free_phase_plane_binned_per_cyst_normalized.png"),
    figure_title="Per-cyst normalized whole-cyst intensity trajectories",
    x_label=display_text("RFP per-cyst normalized whole-cyst intensity"),
    y_label=display_text("YFP per-cyst normalized whole-cyst intensity"),
    bin_width_hours=4.0,
    display_transform="shifted_log",
    shifted_log_axis_quantiles=(0.02, 0.995),
    shifted_log_shift_quantile=0.05,
)

representative_selection_info = representative_positions_by_positive_fraction_lag(
    n_positions=12,
    return_details=True,
)
representative_positions = representative_selection_info["selected_positions"]
representative_selection_path = TABLE_DIR / "05c_positive_fraction_representative_position_selection.tsv"
representative_selection_df = representative_selection_info["selected_df"].copy()
if not representative_selection_df.empty:
    display_time_df(representative_selection_df).to_csv(representative_selection_path, sep="\t", index=False)
    display(Markdown(f"**Representative trajectory selection manifest:** `{representative_selection_path.name}`"))
render_representative_phase_grid(
    joint_df=phase_joint_df,
    figure_path=figure_path("05c_positive_fraction_phase_plane_representative_positions.png"),
    figure_title="Representative cyst positive-fraction trajectories",
    x_label=display_text("RFP fraction"),
    y_label=display_text("YFP fraction"),
    position_labels=representative_positions,
    bin_width_hours=4.0,
)
if not representative_selection_df.empty:
    print("Wrote table:", representative_selection_path)


#### Selected-Timepoint Per-Cyst Vector Summaries

These are exploratory joint-state summaries designed to keep cyst identity explicit without plotting full trajectories.

Each panel is one forward transition between selected clock-time samples:

- each point is one cyst at the starting timepoint
- each arrow shows where that same cyst moves by the next selected timepoint

The five timepoints are chosen from the sampled frames to favor long aggregate transitions while still spanning the movie and keeping per-cyst movement visible.
The positive-fraction version is the main view here.
A density-plus-aggregate-vector alternative is shown below as a cleaner summary.

Selection manifest written in this section:

- `results/tables/05c_positive_fraction_vector_timepoint_selection.tsv`


In [ ]:
selected_vector_info = choose_forward_transition_timepoints(
    phase_joint_df,
    n_timepoints=5,
    candidate_step_hours=2.0,
    min_positions=40,
    return_details=True,
)
selected_vector_times = selected_vector_info["selected_times"]
vector_selection_df = selected_vector_info["transition_df"].copy()
vector_selection_df["candidate_step_hours"] = float(selected_vector_info["candidate_step_hours"])
vector_selection_df["min_positions_required"] = int(selected_vector_info["min_positions"])
vector_selection_df["score_min_aggregate_length"] = float(selected_vector_info["score_min_aggregate_length"])
vector_selection_df["score_total_aggregate_length"] = float(selected_vector_info["score_total_aggregate_length"])
vector_selection_df["score_mean_individual_displacement"] = float(selected_vector_info["score_mean_individual_displacement"])
vector_selection_path = TABLE_DIR / "05c_positive_fraction_vector_timepoint_selection.tsv"
display_time_df(vector_selection_df).to_csv(vector_selection_path, sep="\t", index=False)
display(Markdown(f"**Vector timepoint selection manifest:** `{vector_selection_path.name}`"))
print(
    "Selected vector timepoints (hours):",
    [float(f"{value:.2f}") for value in display_time_hours(selected_vector_times)],
)

render_timepoint_vector_panels(
    joint_df=phase_joint_df,
    figure_path=figure_path("05c_positive_fraction_window_vectors.png"),
    figure_title="Positive-fraction per-cyst vectors across selected timepoints",
    x_label=display_text("RFP positive fraction"),
    y_label=display_text("YFP positive fraction"),
    selected_times=selected_vector_times,
    x_limits=(0.0, 1.0),
    y_limits=(0.0, 1.0),
)

render_timepoint_density_aggregate_panels(
    joint_df=phase_joint_df,
    figure_path=figure_path("05c_positive_fraction_window_density_vectors.png"),
    figure_title="Positive-fraction density and aggregate vectors across selected timepoints",
    x_label=display_text("RFP positive fraction"),
    y_label=display_text("YFP positive fraction"),
    selected_times=selected_vector_times,
    x_limits=(0.0, 1.0),
    y_limits=(0.0, 1.0),
)
print("Wrote table:", vector_selection_path)


#### Position-To-Position Timing Heterogeneity


In [ ]:
theme2_halfmax_specs = [
    {
        "pair_label": "Positive fraction",
        "metric_name_by_reporter": {
            "RFP": "positive_fraction_sigma4",
            "YFP": "positive_fraction_sigma3",
        },
        "normalization_mode": None,
    },
    {
        "pair_label": "Whole-cyst mean intensity (population-wide normalized)",
        "metric_name_by_reporter": {
            "RFP": "organoid_mean_intensity_z",
            "YFP": "organoid_mean_intensity_z",
        },
        "normalization_mode": "global",
    },
    {
        "pair_label": "Whole-cyst mean intensity (per-cyst normalized)",
        "metric_name_by_reporter": {
            "RFP": "organoid_mean_intensity_z",
            "YFP": "organoid_mean_intensity_z",
        },
        "normalization_mode": "per_position",
    },
    {
        "pair_label": "Mean intensity within reporter-positive area (population-wide normalized)",
        "metric_name_by_reporter": {
            "RFP": "positive_mean_intensity_theme2_default_z",
            "YFP": "positive_mean_intensity_theme2_default_z",
        },
        "normalization_mode": "global",
    },
]
theme2_halfmax_rows = []
for spec in theme2_halfmax_specs:
    for position_label, _ in population_metrics.groupby("position_label", sort=True):
        for reporter in ["RFP", "YFP"]:
            metric_name = spec["metric_name_by_reporter"][reporter]
            normalization_mode = spec.get("normalization_mode")
            if normalization_mode is None and metric_name in COMMON_HALFMAX_METRICS:
                halfmax_value = halfmax_time_from_lookup(
                    halfmax_lookup,
                    metric_name,
                    position_label,
                    reporter,
                )
            else:
                halfmax_value = halfmax_time_for_mode(
                    metric_name,
                    position_label,
                    reporter,
                    normalization_mode=normalization_mode or "per_position",
                    clip_normalized=False,
                )
            theme2_halfmax_rows.append(
                {
                    "metric_pair_label": spec["pair_label"],
                    "position_label": position_label,
                    "reporter": reporter,
                    "metric_name": metric_name,
                    "normalization_mode": normalization_mode or "per_position",
                    "halfmax_time_hours": halfmax_value,
                }
            )
theme2_halfmax_df = pd.DataFrame(theme2_halfmax_rows)
theme2_halfmax_path = TABLE_DIR / "05c_halfmax_times.tsv"
display_time_df(theme2_halfmax_df).to_csv(theme2_halfmax_path, sep="\t", index=False)

fig, axes = plt.subplots(1, len(theme2_halfmax_specs), figsize=(5.2 * len(theme2_halfmax_specs), 4.3), constrained_layout=True)
halfmax_bootstrap_rng = np.random.default_rng(20260330)

def bootstrap_ecdf_band(
    values: np.ndarray,
    grid: np.ndarray,
    n_boot: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        nan_grid = np.full_like(grid, np.nan, dtype=float)
        return nan_grid, nan_grid
    sample_idx = halfmax_bootstrap_rng.integers(0, arr.size, size=(n_boot, arr.size))
    sampled = arr[sample_idx]
    ecdf_boot = (sampled[:, :, None] <= grid[None, None, :]).mean(axis=1)
    return (
        np.nanquantile(ecdf_boot, 0.025, axis=0),
        np.nanquantile(ecdf_boot, 0.975, axis=0),
    )

if len(theme2_halfmax_specs) == 1:
    axes = np.asarray([axes])
for ax, spec in zip(axes, theme2_halfmax_specs):
    subset = theme2_halfmax_df.loc[theme2_halfmax_df["metric_pair_label"] == spec["pair_label"]].copy()
    for reporter in ["RFP", "YFP"]:
        values = subset.loc[subset["reporter"] == reporter, "halfmax_time_hours"].to_numpy(dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        ordered = np.sort(values)
        cdf = np.arange(1, ordered.size + 1, dtype=float) / ordered.size
        grid = np.sort(np.unique(ordered))
        ci_lower, ci_upper = bootstrap_ecdf_band(values, grid)
        finite_ci = np.isfinite(grid) & np.isfinite(ci_lower) & np.isfinite(ci_upper)
        if finite_ci.any():
            ax.fill_between(
                grid[finite_ci],
                ci_lower[finite_ci],
                ci_upper[finite_ci],
                step="post",
                color=REPORTER_COLORS[reporter],
                alpha=0.16,
                linewidth=0.0,
                zorder=1,
            )
        ax.step(ordered, cdf, where="post", color=REPORTER_COLORS[reporter], linewidth=2.3, label=reporter_display(reporter))
    ax.set_title(next_plot_title(spec["pair_label"]), fontsize=9.6)
    ax.set_xlabel("Per-position half-max time (hours)")
    set_display_time_axis(ax, "x")
    ax.set_ylabel("Fraction of positions")
    ax.grid(alpha=0.18)
legend_handles = [
    Line2D([0], [0], color=REPORTER_COLORS["RFP"], linewidth=2.3, label=reporter_display("RFP")),
    Line2D([0], [0], color=REPORTER_COLORS["YFP"], linewidth=2.3, label=reporter_display("YFP")),
    Patch(facecolor="0.6", edgecolor="none", alpha=0.20, label="95% bootstrap CI band"),
]
axes[-1].legend(handles=legend_handles, loc="lower right", frameon=False)
halfmax_fig_path = figure_path("05c_halfmax_ecdf.png")
fig.suptitle("Per-position half-max timing", fontsize=11.6)
fig.savefig(halfmax_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", halfmax_fig_path)
print("Wrote table:", theme2_halfmax_path)


In [ ]:
lag_summary_rows = []
rfp_state_specs = [
    {
        "pair_label": "Positive fraction",
        "metric_name_by_reporter": {
            "RFP": "positive_fraction_sigma4",
            "YFP": "positive_fraction_sigma3",
        },
        "normalization_mode": "per_position",
    },
    {
        "pair_label": "Whole-cyst mean intensity (population-wide normalized)",
        "metric_name_by_reporter": {
            "RFP": "organoid_mean_intensity_z",
            "YFP": "organoid_mean_intensity_z",
        },
        "normalization_mode": "global",
    },
    {
        "pair_label": "Whole-cyst mean intensity (per-cyst normalized)",
        "metric_name_by_reporter": {
            "RFP": "organoid_mean_intensity_z",
            "YFP": "organoid_mean_intensity_z",
        },
        "normalization_mode": "per_position",
    },
    {
        "pair_label": "Mean intensity within reporter-positive area (population-wide normalized)",
        "metric_name_by_reporter": {
            "RFP": "positive_mean_intensity_theme2_default_z",
            "YFP": "positive_mean_intensity_theme2_default_z",
        },
        "normalization_mode": "global",
    },
]

fig, axes = plt.subplots(
    3,
    len(theme2_halfmax_specs),
    figsize=(4.4 * len(theme2_halfmax_specs), 8.1),
    constrained_layout=True,
    gridspec_kw={"height_ratios": [1.0, 0.32, 0.32]},
)
timing_bootstrap_rng = np.random.default_rng(20260328)

def bootstrap_median_ci(values: np.ndarray, n_boot: int = 2000) -> tuple[float, float]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (float("nan"), float("nan"))
    sample_idx = timing_bootstrap_rng.integers(0, arr.size, size=(n_boot, arr.size))
    sampled = arr[sample_idx]
    medians = np.nanmedian(sampled, axis=1)
    return (
        float(np.nanquantile(medians, 0.025)),
        float(np.nanquantile(medians, 0.975)),
    )

rfp_state_rows = []
for col_idx, (spec, state_spec) in enumerate(zip(theme2_halfmax_specs, rfp_state_specs)):
    subset = theme2_halfmax_df.loc[theme2_halfmax_df["metric_pair_label"] == spec["pair_label"]].copy()
    pivoted = (
        subset.pivot(index="position_label", columns="reporter", values="halfmax_time_hours")
        .dropna()
        .reset_index()
    )
    scatter_ax = axes[0, col_idx]
    hist_ax = axes[1, col_idx]
    state_ax = axes[2, col_idx]

    x = pivoted["RFP"].to_numpy(dtype=float)
    y = pivoted["YFP"].to_numpy(dtype=float)
    lag_hours = y - x
    lag_ci_lower, lag_ci_upper = bootstrap_median_ci(lag_hours)
    lag_summary_rows.append(
        {
            "metric_pair_label": spec["pair_label"],
            "rfp_metric_name": spec["metric_name_by_reporter"]["RFP"],
            "yfp_metric_name": spec["metric_name_by_reporter"]["YFP"],
            "n_positions": int(len(lag_hours)),
            "median_lag_hours": float(np.nanmedian(lag_hours)),
            "median_lag_ci_lower": lag_ci_lower,
            "median_lag_ci_upper": lag_ci_upper,
            "mean_lag_hours": float(np.nanmean(lag_hours)),
            "fraction_yfp_after_rfp": float(np.mean(lag_hours > 0)),
        }
    )

    lower = float(np.nanmin(np.concatenate([x, y])))
    upper = float(np.nanmax(np.concatenate([x, y])))
    pad = 0.04 * max(upper - lower, 1.0)
    scatter_ax.scatter(x, y, s=28, alpha=0.82, color="#4c78a8")
    scatter_ax.plot([lower, upper], [lower, upper], color="0.25", linestyle="--", linewidth=1.2)
    scatter_ax.set_xlim(lower - pad, upper + pad)
    scatter_ax.set_ylim(lower - pad, upper + pad)
    scatter_ax.set_title(next_plot_title(spec["pair_label"]), fontsize=9.2)
    scatter_ax.set_xlabel(display_text("RFP half-max time (hours)"))
    scatter_ax.set_ylabel(display_text("YFP half-max time (hours)"))
    set_display_time_axis(scatter_ax, "both", crowded=True)
    scatter_ax.grid(alpha=0.18)
    scatter_ax.text(
        0.03,
        0.97,
        f"median lag = {np.nanmedian(lag_hours):.1f} h",
        transform=scatter_ax.transAxes,
        ha="left",
        va="top",
        fontsize=8.7,
        bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.9},
    )

    bin_width_hours = 16.0
    lag_min = float(np.nanmin(lag_hours))
    lag_max = float(np.nanmax(lag_hours))
    edge_offset = 0.5 * bin_width_hours
    left_edge = edge_offset + bin_width_hours * np.floor((lag_min - edge_offset) / bin_width_hours)
    right_edge = edge_offset + bin_width_hours * np.ceil((lag_max - edge_offset) / bin_width_hours)
    lag_bins = np.arange(left_edge, right_edge + bin_width_hours, bin_width_hours)
    if lag_bins.size < 2:
        lag_bins = np.array([left_edge, left_edge + bin_width_hours], dtype=float)
    hist_ax.hist(lag_hours, bins=lag_bins, color="#e0ad00", edgecolor="0.15", alpha=0.82)
    hist_ax.axvline(0.0, color="0.25", linestyle="--", linewidth=1.2)
    hist_ax.axvline(float(np.nanmedian(lag_hours)), color="#d62728", linestyle="-", linewidth=1.5)
    hist_ax.set_xlabel(display_text("YFP half-max minus RFP half-max (hours)"))
    hist_ax.set_ylabel("Positions")
    hist_ax.grid(alpha=0.18)
    if col_idx == 0:
        hist_ax.legend(
            [
                Line2D([0], [0], color="0.25", linestyle="--", linewidth=1.2),
                Line2D([0], [0], color="#d62728", linestyle="-", linewidth=1.5),
            ],
            [display_text("No lag (YFP = RFP)"), "Median lag"],
            loc="upper left",
            frameon=False,
            fontsize=7.0,
        )

    sampled_states = []
    rfp_metric_name = state_spec["metric_name_by_reporter"]["RFP"]
    yfp_metric_name = state_spec["metric_name_by_reporter"]["YFP"]
    normalization_mode = state_spec.get("normalization_mode", "per_position")
    for position_label, _ in population_metrics.groupby("position_label", sort=True):
        if normalization_mode == "per_position" and yfp_metric_name in COMMON_HALFMAX_METRICS:
            yfp_halfmax = halfmax_time_from_lookup(halfmax_lookup, yfp_metric_name, position_label, "YFP")
        else:
            yfp_halfmax = halfmax_time_for_mode(
                yfp_metric_name,
                position_label,
                "YFP",
                normalization_mode=normalization_mode,
                clip_normalized=False,
            )
        if not np.isfinite(yfp_halfmax):
            continue
        rfp_subset = population_metrics.loc[
            (population_metrics["position_label"] == position_label)
            & (population_metrics["reporter"] == "RFP")
        ].sort_values("time_hours")
        time_hours = rfp_subset["time_hours"].to_numpy(dtype=float)
        raw_values = rfp_subset[rfp_metric_name].to_numpy(dtype=float)
        if normalization_mode == "global":
            normalized_state = globally_normalize_trace(
                raw_values,
                metric_name=rfp_metric_name,
                reporter="RFP",
                early_n=8,
                smooth_sigma=4.0,
                clip_normalized=False,
            )
        else:
            normalized_state = normalize_trace(raw_values, early_n=8, smooth_sigma=4.0)
        finite = np.isfinite(time_hours) & np.isfinite(normalized_state)
        if finite.sum() < 12:
            continue
        valid_time = time_hours[finite]
        valid_state = normalized_state[finite]
        if yfp_halfmax < valid_time.min() or yfp_halfmax > valid_time.max():
            continue
        sampled_states.append(float(np.interp(yfp_halfmax, valid_time, valid_state)))

    sampled_states = np.asarray(sampled_states, dtype=float)
    sampled_states = sampled_states[np.isfinite(sampled_states)]
    state_ci_lower, state_ci_upper = (float("nan"), float("nan"))
    if sampled_states.size > 0:
        state_ci_lower, state_ci_upper = bootstrap_median_ci(sampled_states)
        rfp_state_rows.append(
            {
                "metric_pair_label": state_spec["pair_label"],
                "rfp_metric_name": rfp_metric_name,
                "yfp_metric_name": yfp_metric_name,
                "n_positions": int(sampled_states.size),
                "median_rfp_state_at_yfp_halfmax": float(np.nanmedian(sampled_states)),
                "median_rfp_state_ci_lower": state_ci_lower,
                "median_rfp_state_ci_upper": state_ci_upper,
                "mean_rfp_state_at_yfp_halfmax": float(np.nanmean(sampled_states)),
                "fraction_rfp_above_halfmax_when_yfp_hits_halfmax": float(np.mean(sampled_states > 0.5)),
            }
        )
        state_ax.hist(sampled_states, bins=np.linspace(0.0, 1.0, 7), color=REPORTER_COLORS["RFP"], edgecolor="0.15", alpha=0.76)
        state_ax.axvline(0.5, color="0.25", linestyle="--", linewidth=1.2)
        state_ax.axvline(float(np.nanmedian(sampled_states)), color="#111111", linestyle="-", linewidth=1.5)
    state_ax.set_xlabel(display_text("RFP progression at YFP half-max"))
    state_ax.set_ylabel("Positions")
    state_ax.grid(alpha=0.18)
    if col_idx == 0:
        state_ax.legend(
            [
                Line2D([0], [0], color="0.25", linestyle="--", linewidth=1.2),
                Line2D([0], [0], color="#111111", linestyle="-", linewidth=1.5),
            ],
            ["Half-max reference (0.5)", display_text(f"Median sampled RFP state [{state_ci_lower:.2f}, {state_ci_upper:.2f}]")],
            loc="upper left",
            frameon=False,
            fontsize=7.0,
        )

lag_summary_df = pd.DataFrame(lag_summary_rows)
lag_summary_path = TABLE_DIR / "05c_halfmax_lag_summary.tsv"
lag_summary_df.to_csv(lag_summary_path, sep="\t", index=False)
rfp_state_summary_df = pd.DataFrame(rfp_state_rows)
rfp_state_summary_path = TABLE_DIR / "05c_rfp_state_at_yfp_halfmax_summary.tsv"
rfp_state_summary_df.to_csv(rfp_state_summary_path, sep="\t", index=False)

lag_figure_path = figure_path("05c_halfmax_scatter_and_lag.png")
fig.suptitle(display_text("Per-position timing support for YFP following RFP"), fontsize=11.6)
fig.savefig(lag_figure_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
display(display_time_df(lag_summary_df))
display(display_time_df(rfp_state_summary_df))
print("Wrote figure:", lag_figure_path)
print("Wrote table:", lag_summary_path)
print("Wrote table:", rfp_state_summary_path)


#### How Advanced Is RFP When YFP Reaches Half-Max?


In [ ]:
display(display_time_df(rfp_state_summary_df))
print("Wrote table:", rfp_state_summary_path)


#### Threshold Robustness Of The Timing Readout


In [ ]:
threshold_summary_rows = []
threshold_families = [("positive_fraction_sigma{}", "Positive fraction")]

for family_pattern, family_label in threshold_families:
    for sigma_threshold in THRESHOLD_SENSITIVITY_SIGMAS:
        metric_name = family_pattern.format(sigma_threshold)
        if metric_name not in population_metrics.columns:
            continue
        lag_values = []
        rfp_state_values = []
        for position_label, _ in population_metrics.groupby("position_label", sort=True):
            rfp_halfmax = halfmax_time_for_position(population_metrics, metric_name, position_label, "RFP", smooth_sigma=4.0)
            yfp_halfmax = halfmax_time_for_position(population_metrics, metric_name, position_label, "YFP", smooth_sigma=4.0)
            if np.isfinite(rfp_halfmax) and np.isfinite(yfp_halfmax):
                lag_values.append(float(yfp_halfmax - rfp_halfmax))
            if not np.isfinite(yfp_halfmax):
                continue
            rfp_subset = population_metrics.loc[
                (population_metrics["position_label"] == position_label)
                & (population_metrics["reporter"] == "RFP")
            ].sort_values("time_hours")
            time_hours = rfp_subset["time_hours"].to_numpy(dtype=float)
            normalized_state = normalize_trace(rfp_subset[metric_name].to_numpy(dtype=float), early_n=8, smooth_sigma=4.0)
            finite = np.isfinite(time_hours) & np.isfinite(normalized_state)
            if finite.sum() < 12:
                continue
            valid_time = time_hours[finite]
            valid_state = normalized_state[finite]
            if yfp_halfmax < valid_time.min() or yfp_halfmax > valid_time.max():
                continue
            rfp_state_values.append(float(np.interp(yfp_halfmax, valid_time, valid_state)))

        threshold_summary_rows.append(
            {
                "family_label": family_label,
                "metric_name": metric_name,
                "sigma_threshold": sigma_threshold,
                "n_lag_positions": int(len(lag_values)),
                "median_lag_hours": float(np.nanmedian(lag_values)) if lag_values else float("nan"),
                "mean_lag_hours": float(np.nanmean(lag_values)) if lag_values else float("nan"),
                "fraction_yfp_after_rfp": float(np.mean(np.asarray(lag_values, dtype=float) > 0.0)) if lag_values else float("nan"),
                "n_state_positions": int(len(rfp_state_values)),
                "median_rfp_state_at_yfp_halfmax": float(np.nanmedian(rfp_state_values)) if rfp_state_values else float("nan"),
                "mean_rfp_state_at_yfp_halfmax": float(np.nanmean(rfp_state_values)) if rfp_state_values else float("nan"),
            }
        )

threshold_summary_df = pd.DataFrame(threshold_summary_rows)
threshold_summary_path = TABLE_DIR / "05c_threshold_sensitivity_summary.tsv"
threshold_summary_df.to_csv(threshold_summary_path, sep="\t", index=False)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.1), constrained_layout=True)
family_styles = {
    "Positive fraction": "#4c78a8",
}
for family_label, family_df in threshold_summary_df.groupby("family_label"):
    family_df = family_df.sort_values("sigma_threshold")
    axes[0].plot(
        family_df["sigma_threshold"],
        family_df["median_lag_hours"],
        "-o",
        color=family_styles[family_label],
        linewidth=2.2,
        label=family_label,
    )
    axes[1].plot(
        family_df["sigma_threshold"],
        family_df["median_rfp_state_at_yfp_halfmax"],
        "-o",
        color=family_styles[family_label],
        linewidth=2.2,
        label=family_label,
    )

axes[0].axhline(0.0, color="0.45", linestyle="--", linewidth=1.1)
axes[1].axhline(0.5, color="0.45", linestyle="--", linewidth=1.1)
axes[0].set_title(next_plot_title(display_text("Median YFP minus RFP lag vs sigma threshold")), fontsize=10.0)
axes[1].set_title(next_plot_title(display_text("Median RFP progression at YFP half-max vs sigma threshold")), fontsize=10.0)
axes[0].set_xlabel("Sigma threshold")
axes[0].set_ylabel(display_text("Median YFP minus RFP lag (hours)"))
axes[1].set_xlabel("Sigma threshold")
axes[1].set_ylabel(display_text("Median RFP progression at YFP half-max"))
for ax in axes:
    ax.grid(alpha=0.18)
    ax.legend(frameon=False)

threshold_summary_fig_path = figure_path("05c_threshold_sensitivity_summary.png")
fig.suptitle("Threshold sensitivity of the sequential-timing summaries", fontsize=11.6)
fig.savefig(threshold_summary_fig_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
display(display_time_df(threshold_summary_df))
print("Wrote figure:", threshold_summary_fig_path)
print("Wrote table:", threshold_summary_path)


### Main Figure Candidates

These are the strongest nonredundant figures for the Theme 2 story after working through the full notebook.

Together they give:

- the direct population-level clock-time ordering
- the joint-state trajectory view
- the per-position timing support

They are reproduced here at the end so we can evaluate them together without carrying the weaker early intensity candidates.


In [ ]:
candidate_1_source = alternate_figure_path("05c_positive_fraction_trace_summary.png")
candidate_1_path = candidate_figure_path("positive_fraction_trace_summary.png")
candidate_2_source = alternate_figure_path("05c_positive_fraction_window_vectors.png")
candidate_2_path = candidate_figure_path("positive_fraction_window_vectors.png")

for source_path, target_path in [
    (candidate_1_source, candidate_1_path),
    (candidate_2_source, candidate_2_path),
]:
    for suffix in [".png", ".pdf", ".svg"]:
        source_variant = source_path.with_suffix(suffix)
        if source_variant.exists():
            shutil.copy2(source_variant, target_path.with_suffix(suffix))

display(Markdown("#### Candidate 1: Direct Positive-Fraction Ordering In Clock Time"))
display(Image(filename=str(candidate_1_path)))

display(Markdown("#### Candidate 2: Per-Cyst Positive-Fraction Vectors Through State Space"))
display(Image(filename=str(candidate_2_path)))


In [ ]:
positive_fraction_subset = theme2_halfmax_df.loc[
    theme2_halfmax_df["metric_pair_label"] == "Positive fraction"
].copy()
positive_fraction_pivot = (
    positive_fraction_subset.pivot(index="position_label", columns="reporter", values="halfmax_time_hours")
    .dropna()
    .reset_index()
)

positive_fraction_lag_hours = (
    positive_fraction_pivot["YFP"].to_numpy(dtype=float)
    - positive_fraction_pivot["RFP"].to_numpy(dtype=float)
)

positive_fraction_main_path = candidate_figure_path("positive_fraction_timing_support.png")
positive_fraction_bootstrap_rng = np.random.default_rng(20260331)

def bootstrap_ecdf_band_main(
    values: np.ndarray,
    grid: np.ndarray,
    n_boot: int = 2000,
) -> tuple[np.ndarray, np.ndarray]:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        nan_grid = np.full_like(grid, np.nan, dtype=float)
        return nan_grid, nan_grid
    sample_idx = positive_fraction_bootstrap_rng.integers(0, arr.size, size=(n_boot, arr.size))
    sampled = arr[sample_idx]
    ecdf_boot = (sampled[:, :, None] <= grid[None, None, :]).mean(axis=1)
    return (
        np.nanquantile(ecdf_boot, 0.025, axis=0),
        np.nanquantile(ecdf_boot, 0.975, axis=0),
    )

fig, axes = plt.subplots(
    1,
    3,
    figsize=(14.4, 4.1),
    constrained_layout=True,
)
ecdf_ax, scatter_ax, hist_ax = axes

for reporter in ["RFP", "YFP"]:
    values = positive_fraction_subset.loc[
        positive_fraction_subset["reporter"] == reporter,
        "halfmax_time_hours",
    ].to_numpy(dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        continue
    ordered = np.sort(values)
    cdf = np.arange(1, ordered.size + 1, dtype=float) / ordered.size
    grid = np.sort(np.unique(ordered))
    ci_lower, ci_upper = bootstrap_ecdf_band_main(values, grid)
    finite_ci = np.isfinite(grid) & np.isfinite(ci_lower) & np.isfinite(ci_upper)
    if finite_ci.any():
        ecdf_ax.fill_between(
            grid[finite_ci],
            ci_lower[finite_ci],
            ci_upper[finite_ci],
            step="post",
            color=REPORTER_COLORS[reporter],
            alpha=0.16,
            linewidth=0.0,
            zorder=1,
        )
    ecdf_ax.step(
        ordered,
        cdf,
        where="post",
        color=REPORTER_COLORS[reporter],
        linewidth=2.3,
        label=reporter,
    )

ecdf_ax.set_title("Per-position half-max timing", fontsize=9.6)
ecdf_ax.set_xlabel("Half-max time (hours)")
set_display_time_axis(ecdf_ax, "x")
ecdf_ax.set_ylabel("Fraction of positions")
ecdf_ax.grid(alpha=0.18)
ecdf_ax.legend(
    handles=[
        Line2D([0], [0], color=REPORTER_COLORS["RFP"], linewidth=2.3, label=reporter_display("RFP")),
        Line2D([0], [0], color=REPORTER_COLORS["YFP"], linewidth=2.3, label=reporter_display("YFP")),
        Patch(facecolor="0.6", edgecolor="none", alpha=0.20, label="95% bootstrap CI band"),
    ],
    loc="lower right",
    frameon=False,
    fontsize=7.2,
)

x = positive_fraction_pivot["RFP"].to_numpy(dtype=float)
y = positive_fraction_pivot["YFP"].to_numpy(dtype=float)
lower = float(np.nanmin(np.concatenate([x, y])))
upper = float(np.nanmax(np.concatenate([x, y])))
pad = 0.04 * max(upper - lower, 1.0)
scatter_ax.scatter(x, y, s=28, alpha=0.82, color="#4c78a8")
scatter_ax.plot([lower, upper], [lower, upper], color="0.25", linestyle="--", linewidth=1.2)
scatter_ax.set_xlim(lower - pad, upper + pad)
scatter_ax.set_ylim(lower - pad, upper + pad)
scatter_ax.set_title(display_text("RFP vs YFP half-max time"), fontsize=9.6)
scatter_ax.set_xlabel(display_text("RFP half-max time (hours)"))
scatter_ax.set_ylabel(display_text("YFP half-max time (hours)"))
set_display_time_axis(scatter_ax, "both", crowded=True)
scatter_ax.grid(alpha=0.18)
scatter_ax.text(
    0.03,
    0.97,
    f"median lag = {np.nanmedian(positive_fraction_lag_hours):.1f} h",
    transform=scatter_ax.transAxes,
    ha="left",
    va="top",
    fontsize=8.7,
    bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "0.8", "alpha": 0.9},
)

bin_width_hours = 16.0
lag_min = float(np.nanmin(positive_fraction_lag_hours))
lag_max = float(np.nanmax(positive_fraction_lag_hours))
edge_offset = 0.5 * bin_width_hours
left_edge = edge_offset + bin_width_hours * np.floor((lag_min - edge_offset) / bin_width_hours)
right_edge = edge_offset + bin_width_hours * np.ceil((lag_max - edge_offset) / bin_width_hours)
lag_bins = np.arange(left_edge, right_edge + bin_width_hours, bin_width_hours)
if lag_bins.size < 2:
    lag_bins = np.array([left_edge, left_edge + bin_width_hours], dtype=float)
hist_ax.hist(positive_fraction_lag_hours, bins=lag_bins, color="#e0ad00", edgecolor="0.15", alpha=0.82)
hist_ax.axvline(0.0, color="0.25", linestyle="--", linewidth=1.2)
hist_ax.axvline(float(np.nanmedian(positive_fraction_lag_hours)), color="#d62728", linestyle="-", linewidth=1.5)
hist_ax.set_title(display_text("YFP half-max minus RFP half-max"), fontsize=9.6)
hist_ax.set_xlabel("Lag (hours)")
hist_ax.set_ylabel("Positions")
hist_ax.grid(alpha=0.18)
hist_ax.legend(
    [
        Line2D([0], [0], color="0.25", linestyle="--", linewidth=1.2),
        Line2D([0], [0], color="#d62728", linestyle="-", linewidth=1.5),
    ],
    [display_text("No lag (YFP = RFP)"), "Median lag"],
    loc="upper left",
    frameon=False,
    fontsize=7.0,
)

fig.suptitle(display_text("Candidate 3: Positive-fraction timing support for YFP following RFP"), fontsize=11.6)
fig.savefig(positive_fraction_main_path, dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
print("Wrote figure:", positive_fraction_main_path)
